# H&N vs Lung – Rybrevant IV, FASPRO, Lazcluze, and Tagrisso account segmentation, past 52 weeks (L2/L3)

**What changed vs the FASPRO YTD notebook**

|  | FASPRO YTD notebook | This notebook |
| --- | --- | --- |
| Products for the Lung universe | Rybrevant FASPRO only | **Rybrevant IV \+ Rybrevant FASPRO \+ Lazcluze \+ Tagrisso** |
| Window | YTD 2026 (2026\-01\-01 → 2026\-01\-11) | **Past 52 weeks** ending 2026\-11\-01 (2025\-10\-24 → 2026\-11\-01) |
| Chargemaster | CY\-2026 annual rows | **Monthly rows** overlapping the 52 weeks; overlapping periods included in full |
| McKesson | FASPRO alerts only | **All qualifying Rybrevant alerts \+ explicitly named Lazcluze alerts**; no standalone Tagrisso alert rule |
| CareSet | Not included | **CareSet Lung records**, mapped directly from OPAL L0 to L2/L3; overlapping months included in full |
| MX / RX measures | Claim\-event measures | **Distinct patients per account across 52 weeks**, with Tagrisso reported separately |
| Tagrisso claims evidence | Not included | Controlled by **INCLUDE\_TAGRISSO\_CLAIMS\_IN\_LUNG**, currently True; separate patient counts remain available regardless |
| DDD sales split weekly | FASPRO | Setting SPLIT\_PRODUCTS (default **FASPRO**); other products can be added |

**Universes**

| Universe | Sources |
| --- | --- |
| **H&N** | LE.AI H&N cohort (Rybrevant\_HNSCC\_HV\_CLAIMS) \+ Chargemaster HNSCC encounters \+ qualifying McKesson Rybrevant alerts with an H&N diagnosis |
| **Lung** | DDD, Xponent, MX claims, RX claims, McKesson EMR, Chargemaster NSCLC, and CareSet Lung records |

**Product scope:** DDD and Xponent include **Rybrevant IV, FASPRO, Lazcluze, and Tagrisso**. MX/RX retain all four products, but Tagrisso claims contribute to Lung presence only when the claims\-evidence flag is enabled. McKesson includes Rybrevant/Lazcluze alerts; Chargemaster and CareSet do not identify a product.

**Claims patient\-count columns**

| Column | Account\-level measure over 52 weeks |
| --- | --- |
| **MX** | Distinct MX patients across Rybrevant IV, FASPRO, and Lazcluze, excluding Tagrisso |
| **RX** | Distinct RX patients across those same non\-Tagrisso products, after the existing MX patient/product exclusion |
| **MX\_TAGRISSO** | Distinct Tagrisso MX patients |
| **RX\_TAGRISSO** | Distinct Tagrisso RX patients, after the existing MX patient/product exclusion |

Both cleaned patient\-ID components, dv1 and dv2, are required. Counts are calculated after account mapping across the full window, not by adding weekly or product\-level distinct counts.

The RX exclusion uses the same **patient \+ product** across available MX history, not just the 52\-week window or the same account. The four patient\-count columns are **not additive** because patients can overlap across product groups, sources, and accounts.

**Account segment (52 weeks):** both → **H&N \+ Lung**; H&N only → **H&N only**; Lung only → **Lung only**. Accounts with neither signal are excluded.

Segmentation is based on **historical evidence presence**, not patient\-share allocation or confirmation of the indication of an individual purchase. LE.AI retains its **two\-year H&N cohort and patient\-volume measure**.

**Weekly DDD rule:** for each account and week with DDD activity for a SPLIT\_PRODUCTS product, the account is **Lung present** if:

- it has qualifying **non\-DDD Lung evidence anywhere in the full 52\-week window**, including CareSet and eligible Tagrisso activity, **or**
- it bought **any configured Lung product in DDD: Rybrevant IV, FASPRO, Lazcluze, or Tagrisso, during the 13 weeks before** that week.

The lookback excludes the current week. Non\-DDD evidence is retrospective across the full window, so weekly and account\-level segments can differ. Weekly rows use weeks whose **Monday falls inside the window** and retain signed DDD amounts; account\-level DDD measures use positive activity dated within the window.

**Levels:** L0 HCP · L1 outlet · L2 hospital/account · L3 larger account. HCP/outlet records use the selected OPAL **L2\_L3** mapping, representing L2 when available, otherwise L3; directly matched L2/L3 IDs map to themselves. Unmapped activity is excluded from mapped account segmentation, while qualifying unmapped DDD activity remains in the weekly output.

Sections: 1 setup · 2 coverage checks · 3 lookups · 4 H&N universe · 5 Lung activity \+ distinct claims patients · 6 account segments (52 weeks) · 7 weekly DDD split · 8 insights and scorecard · 9 figures · 10 account examples and source breadth · 11 HCP and account profiles · 12 save



## 1. Setup

### What this cell does: settings, helpers, and loading
- **Settings:**
  - `DATASETS`: input dataset names for LE.AI, MX, RX, customer and OPAL lookups, DDD, Xponent, McKesson, Chargemaster, and CareSet.
  - `END_DATE` / `WINDOW_WEEKS`: 52 weeks ending 2026-10-22; calculates `START_DATE` as 2025-10-24.
  - `RYB_NDC`: configured NDC mapping for **Rybrevant IV, FASPRO, Lazcluze, and Tagrisso**.
  - `LUNG_PRODUCTS`: products eligible to provide **Lung evidence**: IV + FASPRO + Lazcluze + Tagrisso.
  - `CLAIMS_LUNG_PRODUCTS`: retains all four products during MX/RX processing.
  - `CLAIMS_NON_TAGRISSO_PRODUCTS`: IV + FASPRO + Lazcluze, used for the separate **`MX` / `RX` distinct patient counts**.
  - `CLAIMS_TAGRISSO_PRODUCTS`: Tagrisso, used for **`MX_TAGRISSO` / `RX_TAGRISSO` distinct patient counts**.
  - `INCLUDE_TAGRISSO_CLAIMS_IN_LUNG`: currently `True`; controls whether Tagrisso MX/RX claims provide Lung evidence. Separate Tagrisso patient counts remain available either way; DDD/Xponent Tagrisso evidence is unaffected.
  - `SPLIT_PRODUCTS`: DDD products split weekly in section 7. Default: FASPRO.
  - `LOOKBACK_WEEKS = 13`: weekly rule, DDD only, weeks **before** the sale week.
  - `CM_HNSCC_AS_HN`, `MCKESSON_HN_DX_AS_HN`: currently `True`; Chargemaster HNSCC / qualifying McKesson Rybrevant H&N diagnoses count as H&N.
  - `HN_ACCOUNT_SOURCE`: `"opal"` uses OPAL `L2_L3`; `"cohort"` uses LE.AI’s mapped account when labeled L2/L3, falling back to OPAL.
  - `RECENT_WEEKS = 3`: number of latest available weeks displayed in the weekly account examples.
- **Helpers:**
  - `load()`: reads a Dataiku dataset into a Spark dataframe.
  - `clean()`: trims text and converts empty strings and configured placeholders, such as `"-"`, `"NA"`, and `"NULL"`, to null.
  - `pick()`: finds a column despite differences in spaces, line breaks, punctuation, or case.
  - `week_of()`: Monday of the week.
  - `ryb_product_ndc()`: identifies the product using the configured NDC mapping after removing nonnumeric characters.
  - `ryb_product_name()`: identifies Tagrisso, Lazcluze, FASPRO, or Rybrevant from case-insensitive product-name text, in that priority order.
- **Loading:**
  - Creates or reuses the Spark session and loads all configured datasets into `df`.
  - Prints the dataset name and raw row count for each input.
  - Product filtering, patient counting, and segmentation are performed in later cells.

In [0]:
import datetime as dt
import re
import dataiku
import dataiku.spark as dkuspark
from pyspark.sql import SparkSession, SQLContext, Window
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()
sqlContext = SQLContext(spark.sparkContext)

# ---------------- settings ----------------
DATASETS = {
    "hn_cohort":    "Rybrevant_HNSCC_HV_CLAIMS",
    "mx":           "MX_RYB_CLAIMS_copy",
    "rx":           "RYB_LINK_CLAIMS_copy",
    "dmn":          "dmn_cust_copy",
    "opal":         "OPAL_Affiliation_Levels",
    "opal_1":       "OPAL_Affiliation_L1",
    "ddd":          "RYB_DDD_Sales_copy",
    "xpon":         "RYB_Xpon_Sales_copy",
    "mckesson":     "Mckesson_GPO_DATA_copy",
    "chargemaster": "ChargeMaster_WeeklyReporting_20261012_v2_copy",
    "careset":      "CARESET_LUNG_copy",
}

END_DATE = "2026-10-22"  # Fixed analysis end date
WINDOW_WEEKS = 52
LOOKBACK_WEEKS = 13

# Shared product recognition mapping.
# Tagrisso remains recognized for DDD/Xponent and MX/RX.
# MX/RX patient counts will be reported separately for Tagrisso.
RYB_NDC = {
    "57894050100": "RYBREVANT",
    "57894050101": "RYBREVANT",
    "57894051001": "RYBREVANT FASPRO",
    "57894051401": "RYBREVANT FASPRO",
    "57894051501": "RYBREVANT FASPRO",
    "57894052201": "RYBREVANT FASPRO",
    "57894008060": "LAZCLUZE",
    "57894008090": "LAZCLUZE",
    "57894024030": "LAZCLUZE",
    "57894024099": "LAZCLUZE",
    "00310124830": "TAGRISSO",
    "00310125130": "TAGRISSO",
    "00310134930": "TAGRISSO",
    "00310135030": "TAGRISSO",
    "00310135095": "TAGRISSO",
}

# Overall Lung product scope: retain Tagrisso.
LUNG_PRODUCTS = [
    "RYBREVANT",
    "RYBREVANT FASPRO",
    "LAZCLUZE",
    "TAGRISSO",
]

# MX/RX processing scope: retain all Lung products.
CLAIMS_LUNG_PRODUCTS = LUNG_PRODUCTS.copy()

# Product groups for separate distinct patient counts.
# MX / RX: non-Tagrisso patients.
# MX_TAGRISSO / RX_TAGRISSO: Tagrisso patients.
CLAIMS_NON_TAGRISSO_PRODUCTS = [
    product
    for product in CLAIMS_LUNG_PRODUCTS
    if product != "TAGRISSO"
]
CLAIMS_TAGRISSO_PRODUCTS = ["TAGRISSO"]

# Controls Tagrisso claims evidence in Lung segmentation.
# Separate Tagrisso patient counts remain available either way.
# This flag does not change Tagrisso inclusion in DDD/Xponent.
INCLUDE_TAGRISSO_CLAIMS_IN_LUNG = True

# DDD products split weekly; unchanged.
SPLIT_PRODUCTS = ["RYBREVANT FASPRO"]

CM_HNSCC_AS_HN = True
MCKESSON_HN_DX_AS_HN = True
HN_ACCOUNT_SOURCE = "opal"
RECENT_WEEKS = 3
# -------------------------------------------

end_dt = dt.datetime.strptime(END_DATE, "%Y-%m-%d").date()
START_DATE = (
    end_dt - dt.timedelta(days=7 * WINDOW_WEEKS - 1)
).isoformat()

MISSING = ["", "-", "--", "NA", "N/A", "NULL", "NONE", "NAN"]


def load(name):
    return dkuspark.get_dataframe(sqlContext, dataiku.Dataset(name))


def clean(col):
    """Trim text; convert empty strings and placeholders to null."""
    s = F.trim(col.cast("string"))
    return F.when(~F.upper(s).isin(MISSING), s)


def pick(d, name):
    """Find a column despite spaces, line breaks, punctuation or case."""
    key = lambda x: re.sub(r"[^0-9a-z]", "", x.lower())
    found = {key(x): x for x in d.columns}

    if key(name) not in found:
        raise KeyError(
            f"column '{name}' not found. Available: {d.columns}"
        )

    return F.col(f"`{found[key(name)]}`")


def week_of(col):
    """Return Monday of the week, consistently across sources."""
    return F.to_date(F.date_trunc("week", F.to_date(col)))


ndc_map = F.create_map(
    *[F.lit(x) for kv in RYB_NDC.items() for x in kv]
)


def ryb_product_ndc(col):
    return ndc_map[
        F.regexp_replace(
            F.trim(col.cast("string")),
            "[^0-9]",
            ""
        )
    ]


def ryb_product_name(col):
    n = F.upper(F.coalesce(col, F.lit("")))

    return (
        F.when(n.contains("TAGRISSO"), "TAGRISSO")
         .when(n.contains("LAZCLUZE"), "LAZCLUZE")
         .when(n.contains("FASPRO"), "RYBREVANT FASPRO")
         .when(n.contains("RYBREVANT"), "RYBREVANT")
    )


df = {k: load(v) for k, v in DATASETS.items()}

for k, v in df.items():
    dataset_name = DATASETS[k]
    print(
        f"{k:13s} {dataset_name:48s} rows={v.count():>12,}"
    )

## 2. Date window

### What this cell does: show the 52\-week window and each source's dates

- **Window:** 52 weeks (364 days), from START\_DATE to END\_DATE, inclusive.
- **Source dates:** Shows the earliest and latest raw dates for DDD, Xponent, MX, RX, and McKesson, and whether each source starts on or before the window start.
- **CareSet:** Shows the earliest and latest month\_start\_date, and checks whether the data includes the window’s start month and reaches its end month.
- **Chargemaster:** Lists the available **cadence values and row counts**, not individual time frames. Section 5 uses the **monthly** rows whose reported periods overlap the window.
- **Coverage interpretation:** Date bounds do not confirm continuous coverage, complete boundary months, or product\-specific coverage. These checks use raw inputs before product filtering and patient\-count processing.



In [0]:
print(f"Window: {START_DATE} .. {END_DATE} ({WINDOW_WEEKS} weeks)\n")
for key, colname in [("ddd", "caln_dt"), ("xpon", "caln_dt"), ("mx", "service_date"), ("rx", "fill_date"),
                     ("mckesson", "diagnosis_date")]:
    r = df[key].agg(F.min(F.to_date(F.col(colname))).alias("min"), F.max(F.to_date(F.col(colname))).alias("max")).collect()[0]
    covers = r["min"] is not None and str(r["min"]) <= START_DATE
    print(f"{key:9s} {colname:15s} {r['min']} .. {r['max']}   starts before window: {'YES' if covers else 'NO'}")

# CareSet: monthly Lung claims; check coverage using month boundaries
careset = df["careset"]
r = careset.agg(
    F.min(F.to_date(pick(careset, "month_start_date"))).alias("min"),
    F.max(F.to_date(pick(careset, "month_start_date"))).alias("max")
).collect()[0]

start_month = end_dt.replace(
    year=int(START_DATE[:4]),
    month=int(START_DATE[5:7]),
    day=1
)
end_month = end_dt.replace(day=1)
covers_start = r["min"] is not None and r["min"] <= start_month
covers_end = r["max"] is not None and r["max"] >= end_month

print("\nCareSet monthly coverage:")
print(
    f"careset   month_start_date {r['min']} .. {r['max']}   "
    f"includes window start month: {'YES' if covers_start else 'NO'}   "
    f"reaches window end month: {'YES' if covers_end else 'NO'}"
)
print("Note: date bounds do not confirm every intervening month is present or the latest month is complete.")

cm = df["chargemaster"]
print("\nChargemaster cadences:")
cm.groupBy(pick(cm, "Cadence").alias("cadence")).count().show(truncate=False)

## 3. Lookups: NPI → J&J ID → L2/L3 account

### What this cell does: build the ID lookups used by every source
- **NPI → J&J ID** (dmn_cust): exclude missing NPIs/J&J IDs. If an NPI has several records, keep the latest `efftv_dt`; break ties using the smallest `jnj_id` in string sort order.
- **Any J&J ID → L2/L3 account** (OPAL), highest-priority available mapping wins:
  1. HCP/L0 (column `0`) → its supplied `L2_L3`; if multiple accounts remain, select the smallest account ID in string sort order.
  2. Outlet/L1 (column `1`) → most frequent `L2_L3` in `OPAL_Affiliation_L1`; break ties by account ID.
  3. ID is itself L2 (column `2`) → itself.
  4. ID is itself L3 (column `3`) → itself.
- **Account level:** L2 if the selected account appears in column `2`; otherwise L3 if it appears in column `3`. Missing level labels become `UNKNOWN` later in the account output.
- **CareSet L0 → account:** map `opal_child_id` directly to OPAL column `0`, using its supplied `L2_L3`. Resolve multiple mappings by the smallest account ID; do not use NPI as a fallback.
- **Mapping helpers:** `to_account()` maps J&J IDs; `careset_to_account()` maps CareSet L0 IDs. Both use left joins, retaining unmapped records for later checks.
- **One mapping per ID:** prevents source-row multiplication during account joins. Patient deduplication is handled later, not in these lookups.
- **Checks:** prints the number of NPIs, J&J IDs, and CareSet L0 IDs available in the mapping lookups.

In [0]:
npi_jnj = (df["dmn"].select(clean(F.col("npi_id")).alias("npi"), clean(F.col("jnj_id")).alias("jnj_id"),
                            F.to_date("efftv_dt").alias("efftv_dt"))
           .filter("npi IS NOT NULL AND jnj_id IS NOT NULL")
           .withColumn("rk", F.row_number().over(Window.partitionBy("npi").orderBy(F.desc_nulls_last("efftv_dt"), "jnj_id")))
           .filter("rk = 1").drop("rk", "efftv_dt").cache())

o = df["opal"].select(*[clean(F.col(f"`{x}`")).alias(n) for x, n in
                        [("0", "hcp"), ("1", "l1"), ("2", "l2"), ("3", "l3"), ("L2_L3", "account_id")]])

o_l1 = df["opal_1"].select(*[clean(F.col(f"`{x}`")).alias(n) for x, n in
                             [("1", "l1"), ("2", "l2"), ("3", "l3"), ("4", "l4"), ("5", "l5"), ("L2_L3", "account_id")]])

by_hcp = (o.filter("hcp IS NOT NULL AND account_id IS NOT NULL")
           .select(F.col("hcp").alias("jnj_id"), "account_id", F.lit(1).alias("prio")))

by_l1 = (o_l1.filter("l1 IS NOT NULL AND account_id IS NOT NULL").groupBy(F.col("l1").alias("jnj_id"), "account_id").count()
          .withColumn("rk", F.row_number().over(Window.partitionBy("jnj_id").orderBy(F.desc("count"), "account_id")))
          .filter("rk = 1").select("jnj_id", "account_id", F.lit(2).alias("prio")))

by_l2 = o.filter("l2 IS NOT NULL").select(F.col("l2").alias("jnj_id"), F.col("l2").alias("account_id"), F.lit(3).alias("prio"))

by_l3 = o.filter("l3 IS NOT NULL").select(F.col("l3").alias("jnj_id"), F.col("l3").alias("account_id"), F.lit(4).alias("prio"))

id_to_account = (by_hcp.unionByName(by_l1).unionByName(by_l2).unionByName(by_l3).distinct()
                 .withColumn("rk", F.row_number().over(Window.partitionBy("jnj_id").orderBy("prio", "account_id")))
                 .filter("rk = 1").select("jnj_id", "account_id").cache())

account_level = (by_l2.select("account_id", F.lit("L2").alias("level"))
                 .unionByName(by_l3.select("account_id", F.lit("L3").alias("level")))
                 .groupBy("account_id").agg(F.min("level").alias("account_level")).cache())

# CareSet: opal_child_id is an L0 ID and maps directly to OPAL column "0".
# Use the supplied L2_L3 account_id, consistent with the existing HCP mapping.
# If an L0 ID has multiple accounts, retain the lowest account_id as a string,
# matching the deterministic tie-break used in id_to_account.
# One mapping per opal_child_id prevents multiplication of CareSet record counts.
opal_child_to_account = (by_hcp.select(F.col("jnj_id").alias("opal_child_id"), "account_id").distinct()
                         .withColumn("rk", F.row_number().over(
                             Window.partitionBy("opal_child_id").orderBy("account_id")))
                         .filter("rk = 1").select("opal_child_id", "account_id").cache())

def to_account(d, id_col):
    return d.join(id_to_account.withColumnRenamed("jnj_id", id_col), id_col, "left")

def careset_to_account(d):
    """Map CareSet L0 opal_child_id directly to its selected OPAL L2/L3 account."""
    return (d.withColumn("opal_child_id", clean(pick(d, "opal_child_id")))
             .join(opal_child_to_account, "opal_child_id", "left"))

print("NPIs in lookup:", f"{npi_jnj.count():,}", "| J&J IDs resolvable to an account:", f"{id_to_account.count():,}")
print("CareSet L0 IDs resolvable to an account:", f"{opal_child_to_account.count():,}")

## 4. H&N universe (LE.AI)

### What this cell does: put the LE.AI H&N HCPs on their L2/L3 accounts
- `"-"` in `J&J ID` is treated as missing; the J&J ID then comes from dmn_cust via the NPI.
- Account per HCP: `HN_ACCOUNT_SOURCE` (`"opal"` = OPAL `L2_L3`).
- `hn_accounts`: one row per account with the number of H&N HCPs and their H&N treating patients (LE.AI, 2 years).

In [0]:
h = df["hn_cohort"]
coh = (h.select(clean(pick(h, "NPI")).alias("npi"), clean(pick(h, "J&J ID")).alias("cohort_jnj_id"),
                clean(pick(h, "Mapped Account JNJ ID")).alias("cohort_account_id"),
                F.upper(clean(pick(h, "Mapped Account JNJ Level"))).alias("cohort_account_level"),
                clean(pick(h, "Mapped Account Name")).alias("cohort_account_name"),
                clean(pick(h, "Treating Patient Volume - HV (2 years)")).cast("double").alias("hn_pts_2y"))
         .join(npi_jnj.select("npi", F.col("jnj_id").alias("dmn_jnj_id")), "npi", "left")
         .withColumn("jnj_id", F.coalesce("cohort_jnj_id", "dmn_jnj_id")))
coh = to_account(coh, "jnj_id").withColumnRenamed("account_id", "opal_account_id")
coh = coh.withColumn("cohort_account", F.when(F.col("cohort_account_level").isin("L2", "L3"), F.col("cohort_account_id")))
acc_col = F.col("opal_account_id") if HN_ACCOUNT_SOURCE == "opal" else F.coalesce("cohort_account", "opal_account_id")
hcp_acc = (coh.withColumn("account_id", acc_col).filter("account_id IS NOT NULL")
              .groupBy("account_id", "npi").agg(F.max("hn_pts_2y").alias("hn_pts_2y")))
hn_accounts = (hcp_acc.groupBy("account_id")
                      .agg(F.countDistinct("npi").alias("hn_hcps"), F.sum("hn_pts_2y").alias("hn_pts_2y")).cache())
print("LE.AI HCPs placed on an account:", hcp_acc.select("npi").distinct().count(), "of", coh.select("npi").distinct().count())
print("H&N accounts:", f"{hn_accounts.count():,}")

## 5.Lung activity by source

### What this cell does: collect Lung activity and distinct claims patients, mapped to accounts
- **Unified activity table:** `activity` contains account, day, week, source, product, amount, J&J ID, and patient ID where available.
- **DDD:** recognizes IV, FASPRO, Lazcluze, and Tagrisso from `prod_nm`; buyer → account; amount = equivalized units. Retains pre-window history for the weekly DDD lookback.
- **Xponent:** recognizes the same four products from `prod_nm`; HCP → account; amount = `equ_trx`.
- **MX claims:**
  - identifies all four products using configured NDCs; `rendering_npi` → J&J ID → account;
  - requires both cleaned `dv1` and `dv2` to construct a valid patient ID;
  - deduplicates by patient + day + NDC + rendering NPI, preserving different provider associations before account mapping;
  - `activity.amount` = one per retained claim event, **not confirmed infusions or distinct patients**.
- **RX claims:**
  - identifies the same four products using configured NDCs; `prescriber_npi` → J&J ID → account;
  - excludes patients present in MX for the **same product**, using available MX history, not only the 52-week window or the same account;
  - deduplicates by `claim_id` + product;
  - `activity.amount` = one per retained claim event.
- **Distinct claims patient counts:** creates `claims_patient_counts_52w`, one row per mapped account:
  - **`MX` / `RX`:** distinct patients across IV, FASPRO, and Lazcluze, excluding Tagrisso;
  - **`MX_TAGRISSO` / `RX_TAGRISSO`:** separate distinct Tagrisso patients;
  - counts directly across the full 52-week window after account mapping; RX counts follow the MX patient/product exclusion;
  - patients can overlap across columns and accounts, so these counts are **not additive**.
- **Tagrisso claims evidence:**
  - `INCLUDE_TAGRISSO_CLAIMS_IN_LUNG` controls whether Tagrisso MX/RX rows remain in eligible `activity`;
  - separate Tagrisso patient counts remain available regardless of the flag;
  - DDD/Xponent Tagrisso activity is unaffected;
  - claims with invalid patient IDs do not contribute to claims-based Lung evidence.
- **McKesson:**
  - includes in-window alerts containing `RYB` or explicitly containing `LAZCLUZE`; `physician_npi` → account; amount = one per alert row;
  - combination alerts count once, retaining the Rybrevant classification;
  - qualifying Rybrevant alerts with `HNSCC`, `HEAD`, or `NECK` in the diagnosis also mark H&N presence;
  - no standalone Tagrisso alert rule is added.
- **Chargemaster:**
  - uses **monthly** rows overlapping the 52-week window, summed per NPI; NPI → account;
  - NSCLC encounters provide Lung activity; HNSCC encounters are retained separately for H&N evidence;
  - uses a no-product placeholder and null day/week.
- **CareSet:**
  - uses positive monthly `num_of_records` for months overlapping the window;
  - maps `opal_child_id` directly from OPAL L0 to the selected L2/L3 account, without an NPI fallback;
  - sums records by account and original L0 ID; retains unmapped records for diagnostics;
  - provides Lung evidence without product attribution; uses a no-product placeholder and null day/week.
- **Activity filtering:** retains positive amounts. Dated sources qualify through actual dates within the window; prefiltered CareSet/Chargemaster rows qualify through monthly overlap. Overlapping monthly counts are included in full.
- **Checks:** prints in-window rows, activity amounts, mapping percentages, and first/last weeks by source and product; displays account-level claims patient counts and Chargemaster/CareSet diagnostics.

In [0]:
in_window = lambda c: F.to_date(c).between(F.lit(START_DATE), F.lit(END_DATE))
parts = []

# DDD (all history kept: the 13-week look-back needs weeks before the window)
# Product helper recognizes Rybrevant IV, FASPRO, Lazcluze, and Tagrisso.
ddd = (df["ddd"].withColumn("product", ryb_product_name(F.col("prod_nm"))).filter(F.col("product").isin(LUNG_PRODUCTS))
                .select(clean(F.col("jnj_id")).alias("jnj_id"), F.to_date("caln_dt").alias("day"),
                        week_of("caln_dt").alias("week"), "product", F.col("equiv_uns").cast("double").alias("amount")))
ddd = to_account(ddd, "jnj_id").cache()
parts.append(ddd.select("account_id", "day", "week", F.lit("DDD").alias("source"), "product", "amount", "jnj_id"))

# Xponent
xp = (df["xpon"].withColumn("product", ryb_product_name(F.col("prod_nm"))).filter(F.col("product").isin(LUNG_PRODUCTS))
                .select(clean(F.col("jnj_id")).alias("jnj_id"), F.to_date("caln_dt").alias("day"), week_of("caln_dt").alias("week"),
                        "product", F.col("equ_trx").cast("double").alias("amount")))
parts.append(to_account(xp, "jnj_id").select("account_id", "day", "week", F.lit("XPONENT").alias("source"), "product", "amount", "jnj_id"))

# Claims patient ID: require both components to avoid partial/empty IDs.
# Invalid patient IDs remain null and do not contribute to distinct patient counts.
claims_patient_id = F.when(
    clean(F.col("dv1")).isNotNull() & clean(F.col("dv2")).isNotNull(),
    F.concat_ws("|", clean(F.col("dv1")), clean(F.col("dv2")))
)

# MX claims
# Retain all eligible claims products, including Tagrisso.
mx = (df["mx"].withColumn("product", ryb_product_ndc(F.col("ndc"))).filter(F.col("product").isin(CLAIMS_LUNG_PRODUCTS))
              .select(claims_patient_id.alias("patient_id"), F.to_date("service_date").alias("day"),
                      "ndc", "product", clean(F.col("rendering_npi")).alias("npi")))

# Retain the original patient lookup for compatibility.
mx_patients = mx.filter("patient_id IS NOT NULL").select("patient_id").distinct()

# Product-specific lookup prevents MX activity for one product from excluding RX for another.
# Preserve the existing exclusion across available MX history, not only the 52-week window.
# Null patient IDs are excluded from the lookup.
mx_patient_products = (mx.filter("patient_id IS NOT NULL")
                        .select("patient_id", "product").distinct())

# Preserve patient IDs and provider associations for account-level distinct counting.
# Include NPI in event deduplication so a patient seen at different accounts is not
# arbitrarily assigned to only one provider before account mapping.
mx_inf = (mx.dropDuplicates(["patient_id", "day", "ndc", "npi"])
            .join(npi_jnj, "npi", "left")
            .select("jnj_id", "day", week_of("day").alias("week"), "product",
                    F.lit(1.0).alias("amount"), "patient_id"))

mx_acc = to_account(mx_inf, "jnj_id").cache()
parts.append(mx_acc.select("account_id", "day", "week", F.lit("MX").alias("source"),
                           "product", "amount", "jnj_id", "patient_id"))

# RX claims (exclude patients present in MX for the same product only)
# Retain Tagrisso and preserve patient IDs for separate distinct patient counts.
# Preserve the existing claim_id/product deduplication.
rx = (df["rx"].withColumn("product", ryb_product_ndc(F.col("ndc"))).filter(F.col("product").isin(CLAIMS_LUNG_PRODUCTS))
              .select(claims_patient_id.alias("patient_id"), "claim_id", F.to_date("fill_date").alias("day"),
                      "product", clean(F.col("prescriber_npi")).alias("npi"))
              .join(mx_patient_products, ["patient_id", "product"], "left_anti")
              .dropDuplicates(["claim_id", "product"])
              .join(npi_jnj, "npi", "left")
              .select("jnj_id", "day", week_of("day").alias("week"), "product",
                      F.lit(1.0).alias("amount"), "patient_id"))

rx_acc = to_account(rx, "jnj_id").cache()
parts.append(rx_acc.select("account_id", "day", "week", F.lit("RX").alias("source"),
                           "product", "amount", "jnj_id", "patient_id"))

# Distinct patients per selected L2/L3 account across the entire 52-week window.
# Count directly across dates and products; do not sum product/week distinct counts.
# RX counts are after the existing MX patient/product exclusion.
claims_patients_52w = (
    mx_acc.select("account_id", "day", "product", "patient_id")
          .withColumn("source", F.lit("MX"))
          .unionByName(
              rx_acc.select("account_id", "day", "product", "patient_id")
                    .withColumn("source", F.lit("RX"))
          )
          .filter(in_window(F.col("day")))
          .filter("account_id IS NOT NULL AND patient_id IS NOT NULL")
          .select("account_id", "source", "product", "patient_id")
          .distinct()
          .cache()
)

claims_patient_counts_52w = (
    claims_patients_52w.groupBy("account_id")
    .agg(
        F.countDistinct(F.when(
            (F.col("source") == "MX") & F.col("product").isin(CLAIMS_NON_TAGRISSO_PRODUCTS),
            F.col("patient_id")
        )).alias("MX"),
        F.countDistinct(F.when(
            (F.col("source") == "RX") & F.col("product").isin(CLAIMS_NON_TAGRISSO_PRODUCTS),
            F.col("patient_id")
        )).alias("RX"),
        F.countDistinct(F.when(
            (F.col("source") == "MX") & F.col("product").isin(CLAIMS_TAGRISSO_PRODUCTS),
            F.col("patient_id")
        )).alias("MX_TAGRISSO"),
        F.countDistinct(F.when(
            (F.col("source") == "RX") & F.col("product").isin(CLAIMS_TAGRISSO_PRODUCTS),
            F.col("patient_id")
        )).alias("RX_TAGRISSO")
    )
    .cache()
)

# McKesson (all Rybrevant alerts plus explicitly named Lazcluze alerts)
an = F.upper(F.coalesce(F.col("alert_name"), F.lit("")))
is_ryb_alert = an.contains("RYB")
is_laz_alert = an.contains("LAZCLUZE")

mk_all = df["mckesson"].filter(
    (is_ryb_alert | is_laz_alert) & in_window(F.col("diagnosis_date"))
)

# Preserve the existing H&N definition: H&N diagnoses on Rybrevant alerts.
dx_txt = F.upper(F.coalesce(F.col("diagnosis"), F.lit("")))
mk_hn = (mk_all.filter(is_ryb_alert & (dx_txt.contains("HNSCC") | dx_txt.contains("HEAD") | dx_txt.contains("NECK")))
               .select(clean(F.col("physician_npi")).alias("npi")).join(npi_jnj, "npi", "left"))
mk_hn = to_account(mk_hn, "jnj_id").filter("account_id IS NOT NULL").select("account_id").distinct()

# Combination alerts are counted once, retaining the Rybrevant classification.
# Lazcluze-only alerts receive the LAZCLUZE product label.
mk = (mk_all.select(clean(F.col("physician_npi")).alias("npi"), F.to_date("diagnosis_date").alias("day"),
                    week_of("diagnosis_date").alias("week"),
                    F.when(is_ryb_alert & an.contains("FASPRO"), "RYBREVANT FASPRO")
                     .when(is_ryb_alert & an.contains("IV"), "RYBREVANT")
                     .when(is_ryb_alert, "RYBREVANT (unspecified)")
                     .when(is_laz_alert, "LAZCLUZE").alias("product"),
                    F.lit(1.0).alias("amount"))
            .join(npi_jnj, "npi", "left"))
parts.append(to_account(mk, "jnj_id").select("account_id", "day", "week", F.lit("MCKESSON").alias("source"), "product", "amount", "jnj_id"))

# Chargemaster: monthly rows overlapping the window, summed per NPI
cm = df["chargemaster"]
desc = F.trim(pick(cm, "Time Frame Descriptive"))
p_start = F.to_date(F.regexp_extract(desc, r"(\d{4}-\d{2}-\d{2})\s*to", 1))
p_end = F.to_date(F.regexp_extract(desc, r"to\s*(\d{4}-\d{2}-\d{2})", 1))
cm_rows = (cm.filter(F.upper(pick(cm, "Cadence")).contains("MONTH"))
             .filter((p_end >= F.lit(START_DATE)) & (p_start <= F.lit(END_DATE)))
             .groupBy(clean(pick(cm, "NPI")).alias("npi"))
             .agg(F.sum(clean(pick(cm, "# EncountersNSCLC")).cast("double")).alias("nsclc"),
                  F.sum(clean(pick(cm, "# EncountersHNSCC")).cast("double")).alias("hnscc"))
             .join(npi_jnj, "npi", "left"))
cm_acc = to_account(cm_rows, "jnj_id").cache()
parts.append(cm_acc.select("account_id", F.lit(None).cast("date").alias("day"), F.lit(None).cast("date").alias("week"),
                           F.lit("CHARGEMASTER").alias("source"), F.lit("NO PRODUCT (Chargemaster)").alias("product"),
                           F.col("nsclc").alias("amount"), "jnj_id"))

# CareSet: monthly Lung claims, mapped directly from L0 opal_child_id to L2/L3.
# Include months overlapping the 52-week window, consistent with Chargemaster.
# Boundary-month counts are included in full; exact daily allocation is unavailable.
careset = df["careset"]
careset_rows = (careset.select(
                    clean(pick(careset, "opal_child_id")).alias("opal_child_id"),
                    clean(pick(careset, "npi")).alias("npi"),
                    F.trunc(F.to_date(pick(careset, "month_start_date")), "month").alias("month_start_date"),
                    clean(pick(careset, "num_of_records")).cast("long").alias("num_of_records"))
                .filter(
                    (F.last_day("month_start_date") >= F.lit(START_DATE)) &
                    (F.col("month_start_date") <= F.lit(END_DATE)) &
                    (F.col("num_of_records") > 0)))

# Uses careset_to_account() from block 3; no NPI-to-J&J-ID lookup is used.
# Keep NPI and month in careset_acc for traceability and source-level checks.
careset_acc = careset_to_account(careset_rows).cache()

# Aggregate counts by mapped account and original L0 ID.
# Unmapped L0 IDs are retained for mapping diagnostics.
careset_activity = (careset_acc.groupBy("account_id", "opal_child_id")
                    .agg(F.sum("num_of_records").cast("double").alias("amount")))

# Like Chargemaster, CareSet has already been filtered by monthly overlap.
# Null day/week avoids treating the first of a month as an exact claim date.
# jnj_id stores the original L0 opal_child_id for the common activity schema.
parts.append(careset_activity.select(
    "account_id",
    F.lit(None).cast("date").alias("day"),
    F.lit(None).cast("date").alias("week"),
    F.lit("CARESET").alias("source"),
    F.lit("NO PRODUCT (CareSet Lung)").alias("product"),
    "amount",
    F.col("opal_child_id").alias("jnj_id")
))

# Add a null patient_id to non-claims sources for a consistent union schema.
parts = [
    p if "patient_id" in p.columns else p.withColumn("patient_id", F.lit(None).cast("string"))
    for p in parts
]

activity = parts[0]
for p in parts[1:]:
    activity = activity.unionByName(p)
activity = activity.filter(F.coalesce("amount", F.lit(0.0)) > 0)

# Only claims with valid patient IDs contribute to claims-based Lung evidence.
# This keeps claims presence consistent with the distinct patient counts.
activity = activity.filter(
    (~F.col("source").isin("MX", "RX")) | F.col("patient_id").isNotNull()
)

# Optionally exclude Tagrisso claims from Lung evidence.
# Counts in claims_patient_counts_52w remain available regardless of this flag.
# DDD/Xponent Tagrisso activity is unaffected.
if not INCLUDE_TAGRISSO_CLAIMS_IN_LUNG:
    activity = activity.filter(
        ~(
            F.col("source").isin("MX", "RX") &
            F.col("product").isin(CLAIMS_TAGRISSO_PRODUCTS)
        )
    )

activity = activity.cache()

# Only prefiltered monthly sources are allowed to have null day values.
# Dated sources with missing/invalid dates do not qualify as in-window activity.
in_win = (
    (F.col("source").isin("CHARGEMASTER", "CARESET") & F.col("day").isNull()) |
    F.col("day").between(F.lit(START_DATE), F.lit(END_DATE))
)

# amount remains an event/activity measure here, not a distinct patient count.
# Use claims_patient_counts_52w for the final MX/RX patient-count columns.
(activity.filter(in_win).groupBy("source", "product")
         .agg(F.count("*").alias("rows"), F.round(F.sum("amount"), 1).alias("amount"),
              F.round(100 * F.sum(F.when(F.col("account_id").isNotNull(), F.col("amount")).otherwise(0)) / F.sum("amount"), 1)
               .alias("pct_amount_on_account"),
              F.min("week").alias("first_week"), F.max("week").alias("last_week"))
         .orderBy("source", "product").show(truncate=False))

print("\n52-week distinct claims patients by account:")
claims_patient_counts_52w.orderBy("account_id").show(truncate=False)
print("Note: patient counts are distinct within each account/column; columns are not additive.")

print("Chargemaster HCPs with HNSCC encounters in window:", cm_acc.filter("hnscc > 0").select("npi").distinct().count())

print("CareSet monthly rows overlapping window:", f"{careset_acc.count():,}")
print("CareSet mapped Lung accounts:", f"{careset_acc.filter('account_id IS NOT NULL').select('account_id').distinct().count():,}")
print("CareSet unmapped monthly rows:", f"{careset_acc.filter('account_id IS NULL').count():,}")

print("\nCareSet record counts by mapping status:")
(careset_acc.withColumn("mapping_status", F.when(F.col("account_id").isNotNull(), "MAPPED").otherwise("UNMAPPED"))
            .groupBy("mapping_status")
            .agg(F.count("*").alias("monthly_rows"),
                 F.countDistinct("opal_child_id").alias("l0_ids"),
                 F.sum("num_of_records").alias("num_of_records"))
            .orderBy("mapping_status").show(truncate=False))

## 6. Account segments (past 52 weeks)

### What this cell does: segment mapped L2/L3 accounts over the 52 weeks
- **Lung universe:** positive qualifying in-window activity from DDD, Xponent, MX, RX, McKesson, Chargemaster NSCLC, or CareSet Lung records. Product-based sources include IV, FASPRO, Lazcluze, and Tagrisso according to each source’s rules.
- **Tagrisso claims:** Lung evidence follows `INCLUDE_TAGRISSO_CLAIMS_IN_LUNG`. Tagrisso DDD/Xponent evidence is unaffected.
- **H&N universe:** LE.AI H&N cohort + Chargemaster HNSCC + qualifying McKesson Rybrevant H&N diagnosis, subject to the H&N settings. LE.AI retains its two-year patient-volume measure.
- **Segment:** H&N only / H&N + Lung / Lung only. Accounts with neither signal are excluded.
- **Claims patient counts:** joins the full-window distinct account-level counts from `claims_patient_counts_52w`:
  - `MX` / `RX`: non-Tagrisso patients;
  - `MX_TAGRISSO` / `RX_TAGRISSO`: separate Tagrisso patients;
  - RX counts retain the MX patient/product exclusion. Counts do not independently change account eligibility and are not additive across columns or accounts.
- **DDD product totals:** reports IV, FASPRO, Lazcluze, and Tagrisso separately; these are components of total DDD, not additional source amounts.
- Prints:
  1. H&N universe accounts per source;
  2. segment counts by account level, including `UNKNOWN` where applicable;
  3. source measures and separate DDD product totals, with MX/RX shown as patient counts rather than event totals;
  4. **CareSet contribution:** accounts with CareSet, accounts whose only eligible Lung evidence is CareSet, and CareSet record counts;
  5. accounts with positive Lazcluze or Tagrisso DDD totals, summarized by segment;
  6. **comparison with a reconstructed FASPRO-only YTD definition:** account movements versus the expanded 52-week definition. The comparison uses current H&N evidence and current-window Chargemaster, so it is not an exact rerun of the earlier notebook.

In [0]:
# Lung universe: qualifying in-window activity, including CareSet.
# Tagrisso claims evidence follows INCLUDE_TAGRISSO_CLAIMS_IN_LUNG
# as applied to activity in block 5. DDD/Xponent remain unchanged.
lung_acct = (activity.filter(in_win & F.col("account_id").isNotNull())
             .groupBy("account_id").pivot("source", ["DDD", "XPONENT", "MX", "RX", "MCKESSON", "CHARGEMASTER", "CARESET"])
             .agg(F.round(F.sum("amount"), 2)).fillna(0)
             .withColumn("lung_present", F.lit(True)))

# Preserve actual non-CareSet Lung evidence before replacing claims event totals.
# This respects the Tagrisso claims flag and is independent of reporting counts.
other_lung_sources = ["DDD", "XPONENT", "MX", "RX", "MCKESSON", "CHARGEMASTER"]
lung_acct = (lung_acct.withColumn(
                 "other_lung_present",
                 sum((F.col(x) > 0).cast("int") for x in other_lung_sources) > 0)
             .drop("MX", "RX"))

# DDD product totals: DDD includes IV + FASPRO + Lazcluze + Tagrisso
ddd_prod = (activity.filter(in_win & F.col("account_id").isNotNull() & (F.col("source") == "DDD"))
            .groupBy("account_id").pivot("product", ["RYBREVANT", "RYBREVANT FASPRO", "LAZCLUZE", "TAGRISSO"])
            .agg(F.round(F.sum("amount"), 2)).fillna(0)
            .withColumnRenamed("RYBREVANT", "DDD_IV")
            .withColumnRenamed("RYBREVANT FASPRO", "DDD_FASPRO")
            .withColumnRenamed("LAZCLUZE", "DDD_LAZCLUZE")
            .withColumnRenamed("TAGRISSO", "DDD_TAGRISSO"))

lung_acct = (lung_acct.join(ddd_prod, "account_id", "left")
             .fillna(0, subset=["DDD_IV", "DDD_FASPRO", "DDD_LAZCLUZE", "DDD_TAGRISSO"]))

# H&N universe: existing LE.AI, Chargemaster HNSCC, and McKesson H&N logic
zero = lambda d: d.withColumn("hn_hcps", F.lit(0)).withColumn("hn_pts_2y", F.lit(0.0))
hn_set = hn_accounts.withColumn("hn_src", F.lit("LE.AI"))

if CM_HNSCC_AS_HN:
    hn_set = hn_set.unionByName(zero(cm_acc.filter("hnscc > 0 AND account_id IS NOT NULL").select("account_id").distinct())
                                .withColumn("hn_src", F.lit("CHARGEMASTER_HNSCC")))

if MCKESSON_HN_DX_AS_HN:
    hn_set = hn_set.unionByName(zero(mk_hn).withColumn("hn_src", F.lit("MCKESSON_HN_DX")))

print("H&N universe accounts by source:")
hn_set.groupBy("hn_src").agg(F.countDistinct("account_id").alias("accounts")).show(truncate=False)

hn_set = (hn_set.groupBy("account_id")
                .agg(F.max("hn_hcps").alias("hn_hcps"), F.max("hn_pts_2y").alias("hn_pts_2y"),
                     F.concat_ws(",", F.sort_array(F.collect_set("hn_src"))).alias("hn_sources"))
                .withColumn("hn_present", F.lit(True)).cache())

def segment_of(hn_col, lung_col):
    hn_flag = F.coalesce(F.col(hn_col), F.lit(False))
    lung_flag = F.coalesce(F.col(lung_col), F.lit(False))
    return (F.when(hn_flag & lung_flag, "H&N + Lung")
             .when(hn_flag, "H&N only")
             .when(lung_flag, "Lung only")
             .otherwise("Unclassified"))

# Patient counts come directly from block 5, not from summed activity amounts.
# MX / RX: distinct non-Tagrisso patients per account across 52 weeks.
# MX_TAGRISSO / RX_TAGRISSO: distinct Tagrisso patients over the same window.
# RX counts retain the existing MX patient/product exclusion.
claims_count_cols = ["MX", "RX", "MX_TAGRISSO", "RX_TAGRISSO"]

seg_order = ["H&N only", "H&N + Lung", "Lung only"]
acct_seg = (hn_set.join(lung_acct, "account_id", "full")
            .join(claims_patient_counts_52w.select("account_id", *claims_count_cols), "account_id", "left")
            .join(account_level, "account_id", "left")
            .fillna(False, subset=["hn_present", "lung_present", "other_lung_present"]).fillna(0)
            .withColumn("account_level", F.coalesce("account_level", F.lit("UNKNOWN")))
            .withColumn("segment", segment_of("hn_present", "lung_present"))
            .cache())

# Left-joining counts preserves the segmentation universe.
# If Tagrisso claims evidence is disabled, Tagrisso-only claims accounts
# without any H&N or other Lung evidence remain outside acct_seg.
# Their patient counts are still available in claims_patient_counts_52w.
print("Account segmentation: Rybrevant IV + FASPRO + Lazcluze + Tagrisso and all eligible Lung sources, including CareSet")
print("Tagrisso claims included in Lung evidence:", INCLUDE_TAGRISSO_CLAIMS_IN_LUNG)
print("MX/RX columns are distinct non-Tagrisso patients; MX_TAGRISSO/RX_TAGRISSO are separate distinct Tagrisso patients.")

tbl = acct_seg.groupBy("account_level").pivot("segment", seg_order).count().fillna(0)
tbl = tbl.withColumn("Total", sum(F.col(f"`{x}`") for x in seg_order))
tbl.unionByName(tbl.agg(*[F.sum(F.col(f"`{x}`")).alias(x) for x in seg_order + ["Total"]]).withColumn("account_level", F.lit("ALL"))) \
   .orderBy("account_level").show(truncate=False)

# Source measures retain their own units; do not sum across different sources.
# DDD product columns are components of DDD, not additional source amounts.
# Claims columns are distinct within each account and reporting group.
# Summing them across accounts gives account-patient counts, not globally unique patients.
# Patients can appear in both non-Tagrisso and Tagrisso groups.
(acct_seg.groupBy("segment")
         .agg(F.count("*").alias("accounts"), F.sum("hn_hcps").alias("hn_hcps"),
              *[F.round(F.sum(x), 1).alias(x) for x in
                ["DDD", "DDD_IV", "DDD_FASPRO", "DDD_LAZCLUZE", "DDD_TAGRISSO", "XPONENT"]],
              *[F.sum(x).alias(x) for x in claims_count_cols],
              *[F.round(F.sum(x), 1).alias(x) for x in
                ["MCKESSON", "CHARGEMASTER", "CARESET"]])
         .withColumn("pct_accounts", F.round(100 * F.col("accounts") / F.sum("accounts").over(Window.partitionBy()), 1))
         .orderBy("segment").show(truncate=False))

# CareSet contribution: distinguish CareSet-only Lung evidence from overlapping sources.
# Use actual eligible evidence captured above, not the reporting patient-count columns.
# Tagrisso counts can remain positive even when excluded from Lung evidence.
other_lung_present = F.col("other_lung_present")

print("CareSet contribution and DDD product evidence by account segment:")
(acct_seg.groupBy("segment")
         .agg(
             F.sum((F.col("CARESET") > 0).cast("int")).alias("accounts_with_careset"),
             F.sum(((F.col("CARESET") > 0) & ~other_lung_present).cast("int")).alias("careset_only_lung_accounts"),
             F.round(F.sum("CARESET"), 0).alias("careset_records"),
             F.sum((F.col("DDD_LAZCLUZE") > 0).cast("int")).alias("accounts_with_ddd_lazcluze"),
             F.sum((F.col("DDD_TAGRISSO") > 0).cast("int")).alias("accounts_with_ddd_tagrisso")
         )
         .orderBy("segment").show(truncate=False))

# Comparison: reconstructed FASPRO-only YTD 2026 Lung definition vs expanded 52-week definition.
# CareSet, Lazcluze, and Tagrisso are excluded from the old Lung definition.
# Preserve the original comparison's Chargemaster treatment:
# positive NSCLC activity from monthly rows overlapping the current 52-week window.
# This is not an exact rerun of the previous notebook's Chargemaster inputs or RX exclusions.
old_lung = (activity.filter(
                F.col("account_id").isNotNull() &
                (
                    (
                        (F.col("product") == "RYBREVANT FASPRO") &
                        F.col("day").between(F.lit("2026-01-01"), F.lit(END_DATE))
                    ) |
                    (F.col("source") == "CHARGEMASTER")
                )
            )
            .select("account_id").distinct().withColumn("lung_old", F.lit(True)))

cmp = (acct_seg.select("account_id", "hn_present", "segment").join(old_lung, "account_id", "full")
               .fillna(False, subset=["hn_present", "lung_old"])
               .withColumn("segment", F.coalesce("segment", F.lit("not in table")))
               .withColumn(
                   "segment_faspro_ytd",
                   F.when(~F.col("hn_present") & ~F.col("lung_old"), "Not in previous universe")
                    .otherwise(segment_of("hn_present", "lung_old"))
               ))

print("Accounts: reconstructed FASPRO-only YTD segment (rows) vs expanded 52-week segment (columns)")
print("Expanded definition includes Rybrevant IV, FASPRO, Lazcluze, Tagrisso, and CareSet Lung evidence.")
print("Tagrisso claims evidence follows INCLUDE_TAGRISSO_CLAIMS_IN_LUNG; DDD/Xponent Tagrisso remains included.")
print("Comparison uses current H&N evidence and current-window Chargemaster; it is not an exact historical rerun.")

cmp.groupBy("segment_faspro_ytd").pivot("segment", seg_order + ["not in table"]).count().fillna(0) \
   .orderBy("segment_faspro_ytd").show(truncate=False)

## 7. Weekly DDD split

### What this cell does: put each week's DDD units (SPLIT\_PRODUCTS) into a segment

- **DDD rows:** products in SPLIT\_PRODUCTS (default FASPRO), weeks whose **Monday falls inside the window**, summed per account × week. Retains signed DDD amounts.
- **Lung present that week:**
    - the account has qualifying **non\-DDD Lung evidence anywhere in the full 52\-week window**, including CareSet, **or**
    - it bought **any configured Lung product in DDD: IV, FASPRO, Lazcluze, or Tagrisso**, during the LOOKBACK\_WEEKS (currently 13 weeks) **before** that week.
- **DDD lookback:** excludes the current week and counts distinct positive purchase weeks. Pre\-window DDD history supports the earliest weeks.
- **Tagrisso evidence:** MX/RX evidence follows INCLUDE\_TAGRISSO\_CLAIMS\_IN\_LUNG; Tagrisso DDD/Xponent evidence is unaffected.
- **H&N present:** uses the H&N universe from section 6, including the retained LE.AI two\-year cohort.
- **Segment:** H&N \+ Lung / H&N only / Lung only / Unclassified (no qualifying signal) / Unmapped account (DDD buyer has no selected OPAL account mapping).
- **Interpretation:** non\-DDD evidence is retrospective across the full window, not restricted to information available before each sale week. A current\-week DDD sale alone does not establish Lung presence, so weekly and 52\-week account segments can differ.
- **Patient counts:** full\-window MX, RX, MX\_TAGRISSO, and RX\_TAGRISSO counts remain in the account table; they are not repeated on weekly rows.
- **Outputs:**
    - units, distinct mapped accounts, and account\-weeks per segment;
    - units per week per segment;
    - latest RECENT\_WEEKS available weeks, showing account units, segment, evidence sources, and prior purchase\-week count;
    - latest available week’s account counts by level and segment.



In [0]:
# Weekly DDD sales to split: controlled by SPLIT_PRODUCTS.
# Preserve the existing rule: include weeks whose Monday falls inside the window.
ddd_aw = (ddd.filter(F.col("product").isin(SPLIT_PRODUCTS) &
                    F.col("week").between(F.lit(START_DATE), F.lit(END_DATE)))
             .groupBy("account_id", "week")
             .agg(F.round(F.sum("amount"), 3).alias("ddd_units")))

# Prior DDD Lung evidence: positive purchases of products in LUNG_PRODUCTS,
# currently Rybrevant IV, FASPRO, Lazcluze, and Tagrisso.
# Retain history before START_DATE for the first weeks' lookback.
# Count distinct purchase weeks, not products or individual purchase records.
# The Tagrisso claims flag does not affect DDD evidence.
ddd_hist = (ddd.filter(
                F.col("account_id").isNotNull() &
                F.col("product").isin(LUNG_PRODUCTS) &
                (F.col("amount") > 0))
               .select(F.col("account_id").alias("a_acc"),
                       F.col("week").alias("a_week"))
               .distinct())

look = (ddd_aw.filter("account_id IS NOT NULL")
        .join(ddd_hist, (F.col("account_id") == F.col("a_acc")) &
                        (F.col("a_week") >= F.date_sub("week", LOOKBACK_WEEKS * 7)) &
                        (F.col("a_week") <= F.date_sub("week", 7)), "left")
        .groupBy("account_id", "week")
        .agg(F.count("a_week").alias("ddd_weeks_prior_13w")))

# Non-DDD Lung evidence across the full analysis window.
# Includes eligible Xponent, MX, RX, McKesson, Chargemaster, and CareSet activity.
# Tagrisso claims inclusion is already controlled in activity by block 5.
# in_win handles the prefiltered monthly CareSet/Chargemaster rows.
# This is retrospective window-level evidence, not an as-of-week-only rule.
hist_lung = (activity.filter(
                 (F.col("source") != "DDD") &
                 F.col("account_id").isNotNull() &
                 in_win &
                 (F.col("amount") > 0))
             .groupBy("account_id")
             .agg(F.concat_ws(",", F.sort_array(F.collect_set("source"))).alias("hist_lung_sources")))

# Preserve weekly segmentation separately from the 52-week account segmentation.
# Do not join full-window patient counts onto every weekly row:
# MX, RX, MX_TAGRISSO, and RX_TAGRISSO remain in acct_seg from block 6.
weekly = (ddd_aw.join(look, ["account_id", "week"], "left").join(hist_lung, "account_id", "left")
                .join(hn_set.select("account_id", "hn_present", "hn_hcps", "hn_sources"), "account_id", "left")
                .join(account_level, "account_id", "left")
                .fillna(False, subset=["hn_present"])
                .fillna(0, subset=["hn_hcps", "ddd_weeks_prior_13w"])
                .withColumn("lung_present",
                            F.col("hist_lung_sources").isNotNull() |
                            (F.col("ddd_weeks_prior_13w") > 0))
                .withColumn("segment", F.when(F.col("account_id").isNull(), "Unmapped account")
                                        .when(F.col("hn_present") & F.col("lung_present"), "H&N + Lung")
                                        .when(F.col("hn_present"), "H&N only")
                                        .when(F.col("lung_present"), "Lung only")
                                        .otherwise("Unclassified"))
                .cache())

all_segs = seg_order + ["Unclassified", "Unmapped account"]
print("Products split:", SPLIT_PRODUCTS)
print("Products eligible for prior DDD Lung evidence:", LUNG_PRODUCTS)
print("Prior DDD lookback weeks:", LOOKBACK_WEEKS)
print("Non-DDD Lung evidence includes CareSet and qualifying product activity.")
print("Tagrisso claims included in Lung evidence:", INCLUDE_TAGRISSO_CLAIMS_IN_LUNG)
print("Tagrisso DDD/Xponent evidence remains included regardless of the claims flag.")
print("Non-DDD evidence is evaluated across the full analysis window.")
print("Distinct MX/RX patient counts, including separate Tagrisso columns, remain in acct_seg.")

print("1) DDD units per segment (whole window)")
(weekly.groupBy("segment")
       .agg(F.round(F.sum("ddd_units"), 1).alias("ddd_units"),
            F.countDistinct("account_id").alias("accounts"),
            F.count("*").alias("account_weeks"))
       .withColumn(
           "pct_units",
           F.when(
               F.sum("ddd_units").over(Window.partitionBy()) != 0,
               F.round(100 * F.col("ddd_units") /
                       F.sum("ddd_units").over(Window.partitionBy()), 1)
           ).otherwise(F.lit(0.0)))
       .orderBy("segment").show(truncate=False))

print("2) DDD units per week per segment")
weekly.groupBy("week").pivot("segment", all_segs).agg(F.round(F.sum("ddd_units"), 1)) \
      .fillna(0).orderBy("week").show(60, False)

last_weeks = [r["week"] for r in weekly.select("week").distinct()
             .orderBy(F.desc("week")).limit(RECENT_WEEKS).collect()]

print(f"3) Latest {RECENT_WEEKS} weeks, per account:", [str(w) for w in sorted(last_weeks)])

if last_weeks:
    (weekly.filter(F.col("week").isin(last_weeks))
           .select("week", "account_id", "account_level", "ddd_units", "segment",
                   "hn_sources", "hist_lung_sources", "ddd_weeks_prior_13w")
           .orderBy(F.desc("week"), F.desc("ddd_units")).show(50, False))
else:
    print("No DDD account-weeks found for SPLIT_PRODUCTS in the selected window.")

latest = max(last_weeks) if last_weeks else None
print("4) Latest week", latest, "- accounts per segment by level")

if latest is not None:
    (weekly.filter(F.col("week") == F.lit(latest))
           .groupBy(F.coalesce("account_level", F.lit("UNMAPPED")).alias("account_level"))
           .pivot("segment", all_segs).count().fillna(0)
           .orderBy("account_level").show(truncate=False))
else:
    print("No latest week available.")

## 8. Account segment insights (L2/L3)

### What this cell does: insight table + scorecard
- **One row per account (`ins`):**
  - name from LE.AI where available;
  - number of **eligible Lung sources**, calculated from activity and respecting the Tagrisso claims-evidence flag;
  - Chargemaster-only Lung flag (`weak_lung_chargemaster_only`);
  - CareSet-only Lung flag (`lung_careset_only`);
  - **DDD product mix:** positive totals for IV, FASPRO, Lazcluze, and/or Tagrisso, or `no DDD`;
  - retains the separate account-level `MX`, `RX`, `MX_TAGRISSO`, and `RX_TAGRISSO` patient counts.
- **Scorecard by level × segment:**
  - accounts and % of total accounts;
  - DDD units: total, IV, FASPRO, Lazcluze, and Tagrisso, plus % of total DDD units;
  - DDD units per account;
  - `MX` / `RX`: distinct non-Tagrisso patients per account, summed across accounts;
  - `MX_TAGRISSO` / `RX_TAGRISSO`: separate distinct Tagrisso patients per account, summed across accounts;
  - H&N HCP count and LE.AI **two-year** treating-patient volume;
  - average number of eligible Lung sources;
  - accounts with CareSet, CareSet-only Lung accounts, and CareSet record counts.
- **Product-mix table:** segment × observed IV / FASPRO / Lazcluze / Tagrisso buying pattern. This does **not** establish treatment combinations, switching, or an IV → FASPRO transition.
- **CareSet-only table:** accounts by segment and whether CareSet is their only eligible Lung source.
- **Interpretation:** claims counts are distinct within each account and reporting group, not globally unique across accounts. Do not add the patient-count columns together. RX counts retain the existing MX patient/product exclusion.

In [0]:
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)

SEG_COLORS = {"H&N only": "#4C78A8", "H&N + Lung": "#B279A2", "Lung only": "#54A24B",
              "Unclassified": "#BAB0AC", "Unmapped account": "#E45756"}
LUNG_SRC = ["DDD", "XPONENT", "MX", "RX", "MCKESSON", "CHARGEMASTER", "CARESET"]

acct_names = (coh.filter("cohort_account_id IS NOT NULL")
                 .select(F.col("cohort_account_id").alias("account_id"), F.col("cohort_account_name").alias("account_name"))
                 .dropDuplicates(["account_id"]))

# Count eligible Lung sources directly from activity.
# This respects INCLUDE_TAGRISSO_CLAIMS_IN_LUNG as applied in block 5.
# Tagrisso is part of MX/RX evidence, not an additional source.
# Do not infer source presence from MX/RX reporting columns alone:
# those columns exclude Tagrisso, while separate Tagrisso counts remain available.
lung_source_counts = (activity.filter(
                          in_win &
                          F.col("account_id").isNotNull() &
                          F.col("source").isin(LUNG_SRC) &
                          (F.col("amount") > 0))
                       .groupBy("account_id")
                       .agg(F.countDistinct("source").alias("n_lung_sources")))

# Product mix reflects positive DDD totals during the analysis window.
# It does not establish treatment combinations or chronological switching.
has_iv = F.col("DDD_IV") > 0
has_faspro = F.col("DDD_FASPRO") > 0
has_lazcluze = F.col("DDD_LAZCLUZE") > 0
has_tagrisso = F.col("DDD_TAGRISSO") > 0

# Build all product combinations without maintaining a separate condition for each.
# Preserve the existing label order: IV, FASPRO, LAZCLUZE, TAGRISSO.
ddd_mix = F.concat_ws(" + ",
                     F.when(has_iv, F.lit("IV")),
                     F.when(has_faspro, F.lit("FASPRO")),
                     F.when(has_lazcluze, F.lit("LAZCLUZE")),
                     F.when(has_tagrisso, F.lit("TAGRISSO")))

ddd_product_count = sum(F.coalesce(flag, F.lit(False)).cast("int")
                        for flag in [has_iv, has_faspro, has_lazcluze, has_tagrisso])

# acct_seg already contains the four distinct patient-count columns from block 6.
ins = (acct_seg.join(acct_names, "account_id", "left")
       .join(lung_source_counts, "account_id", "left")
       .fillna(0, subset=["n_lung_sources", "MX", "RX", "MX_TAGRISSO", "RX_TAGRISSO"])
       .withColumn("weak_lung_chargemaster_only",
                   (F.col("n_lung_sources") == 1) & (F.col("CHARGEMASTER") > 0))
       .withColumn("lung_careset_only",
                   (F.col("n_lung_sources") == 1) & (F.col("CARESET") > 0))
       .withColumn("ddd_product_mix",
                   F.when(ddd_product_count == 0, F.lit("no DDD"))
                    .when(ddd_product_count == 1, F.concat(ddd_mix, F.lit(" only")))
                    .otherwise(ddd_mix))
       .cache())

# DDD totals include IV + FASPRO + Lazcluze + Tagrisso.
# Product totals are components of DDD, not additional source amounts.
# CareSet record counts are reported separately, not added to DDD units.
# MX/RX exclude Tagrisso; MX_TAGRISSO/RX_TAGRISSO report it separately.
# Claims counts are distinct per account across the full 52-week window.
# Summing across accounts gives account-patient counts, not globally unique patients.
score = (ins.groupBy("account_level", "segment")
            .agg(F.count("*").alias("accounts"), F.round(F.sum("DDD"), 1).alias("ddd_units"),
                 F.round(F.sum("DDD_IV"), 1).alias("ddd_iv"),
                 F.round(F.sum("DDD_FASPRO"), 1).alias("ddd_faspro"),
                 F.round(F.sum("DDD_LAZCLUZE"), 1).alias("ddd_lazcluze"),
                 F.round(F.sum("DDD_TAGRISSO"), 1).alias("ddd_tagrisso"),
                 F.sum("MX").alias("MX"),
                 F.sum("RX").alias("RX"),
                 F.sum("MX_TAGRISSO").alias("MX_TAGRISSO"),
                 F.sum("RX_TAGRISSO").alias("RX_TAGRISSO"),
                 F.sum("hn_hcps").alias("hn_hcps"),
                 F.round(F.sum("hn_pts_2y"), 0).alias("hn_treating_pts_2y"),
                 F.round(F.avg("n_lung_sources"), 2).alias("avg_lung_sources"),
                 F.sum((F.col("CARESET") > 0).cast("int")).alias("accounts_with_careset"),
                 F.sum(F.col("lung_careset_only").cast("int")).alias("careset_only_lung_accounts"),
                 F.round(F.sum("CARESET"), 0).alias("careset_records"))
            .toPandas())

total_accounts = score["accounts"].sum()
total_ddd_units = score["ddd_units"].sum()

score["pct_accounts"] = (
    (100 * score["accounts"] / total_accounts).round(1)
    if total_accounts > 0 else 0.0
)
score["pct_ddd_units"] = (
    (100 * score["ddd_units"] / total_ddd_units).round(1)
    if total_ddd_units != 0 else 0.0
)
score["units_per_account"] = (
    score["ddd_units"] / score["accounts"].where(score["accounts"] > 0)
).fillna(0).round(2)

print(score.sort_values(["account_level", "segment"]).to_string(index=False))
print("\nMX/RX: distinct non-Tagrisso patients per account over 52 weeks.")
print("MX_TAGRISSO/RX_TAGRISSO: separate distinct Tagrisso patients per account over 52 weeks.")
print("RX counts retain the existing MX patient/product exclusion.")
print("Patient-count columns are not additive across product groups, sources, or accounts.")
print("Tagrisso claims included in Lung evidence:", INCLUDE_TAGRISSO_CLAIMS_IN_LUNG)

print("\nDDD product mix: accounts purchasing IV, FASPRO, Lazcluze, and/or Tagrisso during the window")
ins.groupBy("ddd_product_mix").pivot("segment", seg_order).count().fillna(0) \
   .orderBy("ddd_product_mix").show(20, truncate=False)

print("\nCareSet-only Lung evidence: accounts by segment")
ins.groupBy("lung_careset_only").pivot("segment", seg_order).count().fillna(0) \
   .orderBy("lung_careset_only").show(truncate=False)

### What this cell does: figures

1. **Accounts and DDD units** (IV \+ FASPRO \+ Lazcluze \+ Tagrisso) per segment, stacked by available account level.
2. **Weekly DDD units by product:** IV, FASPRO, Lazcluze, and Tagrisso, including mapped and unmapped buyers and signed amounts. Shows product trends, not confirmed switching or treatment transitions.
3. **Weekly DDD units (****SPLIT\_PRODUCTS****) by segment**, using the weekly classification from section 7. Default: FASPRO only.
4. **Concentration of positive mapped account DDD totals** across all four products: cumulative share of units by ranked account, plus the shares purchased by the top 10, 50, and 100 accounts where available.
5. **H&N \+ Lung accounts:** total account DDD units across all four products versus LE.AI **two\-year treating\-patient volume**, with L2 and L3 plotted separately. The patient\-volume measure is not a distinct 52\-week patient count.



In [0]:
# Account counts and total DDD units by segment and account level.
# Total DDD includes IV + FASPRO + Lazcluze + Tagrisso.
# Separate MX/RX patient-count columns do not affect these charts.
segs = ["H&N only", "H&N + Lung", "Lung only"]
lv = score.pivot_table(index="segment", columns="account_level", values=["accounts", "ddd_units"], aggfunc="sum").fillna(0)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, metric, title in [(axes[0], "accounts", "Accounts per segment"),
                          (axes[1], "ddd_units", "DDD units (IV + FASPRO + Lazcluze + Tagrisso), 52 weeks")]:
    t = lv[metric].reindex(segs).fillna(0)
    t.plot(kind="bar", stacked=True, ax=ax, color=["#1f77b4", "#ff7f0e", "#999999"][:t.shape[1]])
    ax.set_title(title); ax.set_xlabel(""); plt.setp(ax.get_xticklabels(), rotation=0)
    for i, v in enumerate(t.sum(axis=1)):
        ax.text(i, v, f"{v:,.0f}", ha="center", va="bottom")
plt.tight_layout(); plt.show()

# Weekly DDD trend for all configured Lung products.
# Preserve the existing rule: include weeks whose Monday falls inside the window.
# Includes mapped and unmapped buyers and retains signed DDD amounts.
PRODUCT_COLORS = {"RYBREVANT": "#F58518", "RYBREVANT FASPRO": "#4C78A8",
                  "LAZCLUZE": "#54A24B", "TAGRISSO": "#B279A2"}
PRODUCT_LABELS = {"RYBREVANT": "Rybrevant IV", "RYBREVANT FASPRO": "Rybrevant FASPRO",
                  "LAZCLUZE": "Lazcluze", "TAGRISSO": "Tagrisso"}

tr = (ddd.filter(F.col("week").between(F.lit(START_DATE), F.lit(END_DATE)))
         .groupBy("week").pivot("product", LUNG_PRODUCTS).agg(F.sum("amount")).fillna(0)
         .orderBy("week").toPandas().set_index("week"))
fig, ax = plt.subplots(figsize=(13, 4))
for col in LUNG_PRODUCTS:
    if col in tr.columns:
        ax.plot(tr.index, tr[col], label=PRODUCT_LABELS.get(col, col),
                color=PRODUCT_COLORS.get(col), linewidth=2)
ax.set_title("Weekly DDD units: Rybrevant IV, FASPRO, Lazcluze, and Tagrisso (all buyers)")
ax.set_ylabel("equivalized units"); ax.legend()
plt.tight_layout(); plt.show()

# Weekly units by segment remain limited to SPLIT_PRODUCTS.
# Tagrisso can contribute Lung evidence without being included in these units.
# Tagrisso claims evidence follows INCLUDE_TAGRISSO_CLAIMS_IN_LUNG.
# DDD/Xponent Tagrisso evidence is unaffected by that flag.
wk = (weekly.groupBy("week").pivot("segment", list(SEG_COLORS)).agg(F.sum("ddd_units")).fillna(0)
            .orderBy("week").toPandas().set_index("week"))
fig, ax = plt.subplots(figsize=(13, 4))
for col in wk.columns:
    ax.plot(wk.index, wk[col], label=col, color=SEG_COLORS.get(col), linewidth=2)
ax.set_title("Weekly DDD units by segment (" + ", ".join(SPLIT_PRODUCTS) + ")"); ax.set_ylabel("equivalized units")
ax.legend(ncol=5, fontsize=8)
plt.tight_layout(); plt.show()

# Concentration uses mapped positive in-window DDD activity across all Lung products.
p = ins.filter("DDD > 0").select("account_id", "DDD").toPandas().sort_values("DDD", ascending=False)
total_mapped_ddd = p["DDD"].sum()
p["cum_pct"] = 100 * p["DDD"].cumsum() / total_mapped_ddd if total_mapped_ddd > 0 else 0.0

for n in (10, 50, 100):
    if len(p) >= n:
        print(f"Top {n} accounts buy {p.cum_pct.iloc[n - 1]:.1f}% of mapped DDD units (of {len(p):,} buying accounts)")

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, len(p) + 1), p.cum_pct.values); ax.axhline(80, ls="--", color="grey")
ax.set_xlabel("accounts (ranked by DDD units)"); ax.set_ylabel("cumulative % of units")
ax.set_title("Concentration of DDD (all Lung products)")
if p.empty:
    ax.text(0.5, 0.5, "No positive mapped DDD activity",
            transform=ax.transAxes, ha="center", va="center")
plt.tight_layout(); plt.show()

# H&N + Lung scatter uses total DDD across all Lung products.
# hn_pts_2y remains the supplied LE.AI two-year HCP-volume rollup,
# not a distinct 52-week patient count.
sc = ins.filter("segment = 'H&N + Lung'").select("DDD", "hn_pts_2y", "account_level").toPandas()
fig, ax = plt.subplots(figsize=(6, 4))
for lvl, c in [("L2", "#1f77b4"), ("L3", "#ff7f0e")]:
    d = sc[sc.account_level == lvl]
    ax.scatter(d.DDD, d.hn_pts_2y, s=14, alpha=0.6, label=lvl, color=c)
ax.set_xlabel("DDD units (IV + FASPRO + Lazcluze + Tagrisso), 52 weeks")
ax.set_ylabel("H&N treating patients (LE.AI, 2y)")
ax.set_title("H&N + Lung accounts"); ax.legend()
plt.tight_layout(); plt.show()

### What this cell does: actionable account lists

1. **H&N only:** accounts with H&N evidence and no qualifying Lung evidence, ranked by LE.AI two\-year treating\-patient volume, then H&N HCP count.
2. **H&N \+ Lung:** ranked by total DDD units, with separate IV / FASPRO / Lazcluze / Tagrisso totals.
3. **Lung only:** ranked by total DDD units, without qualifying H&N evidence in the analysis inputs.
4. **Source breadth:** accounts by number of eligible Lung sources; separately counts Chargemaster\-only and CareSet\-only Lung accounts. Source breadth is not a validated confidence score.
5. **CareSet\-only Lung evidence:** accounts ranked by CareSet record count; these accounts can still have H&N evidence.
6. **Lazcluze DDD buyers:** accounts with positive Lazcluze DDD totals, ranked by Lazcluze units.
7. **Tagrisso DDD buyers:** accounts with positive Tagrisso DDD totals, ranked by Tagrisso units.
- **Displayed measures:** account details, H&N measures, DDD product totals and mix, source measures, evidence breadth, and H&N sources.
- **Claims columns:** MX / RX show distinct non\-Tagrisso patients per account over 52 weeks; MX\_TAGRISSO / RX\_TAGRISSO show separate Tagrisso patients. RX counts retain the MX patient/product exclusion.
- **Tagrisso flexibility:** separate patient counts remain visible even when Tagrisso claims are excluded from Lung evidence. Patient\-count columns are not additive.



In [0]:
# MX/RX: distinct non-Tagrisso patients per account over 52 weeks.
# MX_TAGRISSO/RX_TAGRISSO: separate distinct Tagrisso patients.
# RX counts retain the existing MX patient/product exclusion.
cols = ["account_id", "account_name", "account_level", "hn_hcps", "hn_pts_2y",
        "DDD", "DDD_IV", "DDD_FASPRO", "DDD_LAZCLUZE", "DDD_TAGRISSO", "ddd_product_mix",
        "XPONENT", "MX", "RX", "MX_TAGRISSO", "RX_TAGRISSO",
        "MCKESSON", "CHARGEMASTER", "CARESET",
        "n_lung_sources", "hn_sources"]

print("1) H&N only - no qualifying Lung evidence in the analysis inputs")
(ins.filter("segment = 'H&N only'")
    .orderBy(F.desc("hn_pts_2y"), F.desc("hn_hcps"), "account_id")
    .select(*cols).show(20, False))

print("2) H&N + Lung - largest DDD buyers across IV, FASPRO, Lazcluze, and Tagrisso with H&N evidence")
(ins.filter("segment = 'H&N + Lung'")
    .orderBy(F.desc("DDD"), "account_id")
    .select(*cols).show(20, False))

print("3) Lung only - largest DDD buyers across IV, FASPRO, Lazcluze, and Tagrisso without H&N evidence")
(ins.filter("segment = 'Lung only'")
    .orderBy(F.desc("DDD"), "account_id")
    .select(*cols).show(20, False))

print("4) Source breadth - accounts by number of qualifying Lung sources, including CareSet")
(ins.filter("segment != 'H&N only'")
    .groupBy("segment").pivot("n_lung_sources", list(range(1, len(LUNG_SRC) + 1)))
    .count().fillna(0).orderBy("segment").show(truncate=False))

print("Accounts whose only Lung source is Chargemaster (no product column):",
      ins.filter("weak_lung_chargemaster_only").count())

print("Accounts whose only Lung source is CareSet (no product column):",
      ins.filter("lung_careset_only").count())

print("5) CareSet-only Lung evidence - accounts ranked by CareSet record count")
(ins.filter("lung_careset_only")
    .orderBy(F.desc("CARESET"), "account_id")
    .select("segment", *cols).show(20, False))

print("6) Accounts with positive Lazcluze DDD totals, ranked by Lazcluze units")
(ins.filter("DDD_LAZCLUZE > 0")
    .orderBy(F.desc("DDD_LAZCLUZE"), "account_id")
    .select("segment", *cols).show(20, False))

print("7) Accounts with positive Tagrisso DDD totals, ranked by Tagrisso units")
(ins.filter("DDD_TAGRISSO > 0")
    .orderBy(F.desc("DDD_TAGRISSO"), "account_id")
    .select("segment", *cols).show(20, False))

## 9. Segment profiles

### What this cell does: describe who is in each segment
1. **Specialty:** specialty mix of LE.AI H&N HCPs, product-associated HCPs, and CareSet Lung HCPs, using the latest-effective `dmn_cust` record’s `prim_spclt_nm`. Product-associated HCPs come from eligible MX, RX, Xponent, and McKesson activity, not necessarily confirmed prescribing.
2. **HCPs per account and overlap:** average LE.AI H&N, product-associated, and CareSet Lung HCP counts; identifies HCPs appearing in both LE.AI and product-associated activity, or both LE.AI and CareSet, at the **same account and J&J ID**. Product-associated activity follows the Tagrisso claims-evidence flag.
3. **Account type:** account-level `cust_typ_nm` from the latest-effective customer record; unmatched or missing types are shown separately.
4. **Practice setting:** most frequent LE.AI practice-setting label per account, breaking ties alphabetically. Reported for H&N only and H&N + Lung accounts.
5. **MX care-setting profile:** claim events, distinct patients, institutional-line percentage, and TB/UD modifier-line percentage by segment and product: **IV, FASPRO, Lazcluze, and Tagrisso**. Uses valid patient IDs and retains Tagrisso for descriptive reporting regardless of the claims-evidence flag. Events are not assumed infusions; modifier flags are proxies, not confirmed 340B status. Patients are distinct within each segment × product and are not additive across products.
6. **DDD purchase activity:** positive mapped purchases across all four products, showing first observed week within the window, active weeks, and the latest 13 weeks versus the previous 13. Periods are anchored to the latest mapped purchase week in the window. “First observed” does not mean first-ever adoption, and comparisons use positive purchases rather than net sales including returns.

In [0]:
seg_of = ins.select("account_id", "segment")
win = F.col("day").between(F.lit(START_DATE), F.lit(END_DATE))

# Latest-effective customer record for specialty and customer type.
dmn_info = (df["dmn"].select(
                clean(F.col("jnj_id")).alias("jnj_id"),
                clean(F.col("prim_spclt_nm")).alias("specialty"),
                clean(F.col("cust_typ_nm")).alias("cust_type"),
                F.to_date("efftv_dt").alias("efftv_dt"))
            .filter("jnj_id IS NOT NULL")
            .withColumn("rk", F.row_number().over(
                Window.partitionBy("jnj_id").orderBy(
                    F.desc_nulls_last("efftv_dt"),
                    F.asc_nulls_last("specialty"),
                    F.asc_nulls_last("cust_type"))))
            .filter("rk = 1").drop("rk", "efftv_dt").cache())

hn_hcp = (coh.withColumn("account_id", acc_col)
             .filter("account_id IS NOT NULL AND jnj_id IS NOT NULL")
             .select("account_id", "jnj_id", "npi").distinct())

# Product-associated HCPs from eligible MX, RX, Xponent, and McKesson activity.
# Use activity so claims follow the valid patient-ID rule and Tagrisso flag.
# Tagrisso Xponent evidence remains included regardless of the claims flag.
# These are source-associated HCPs, not necessarily confirmed prescribers.
# DDD buyers are not treated as HCPs.
writers = (activity.filter(
                F.col("source").isin("MX", "RX", "XPONENT", "MCKESSON") &
                win &
                (F.col("amount") > 0) &
                F.col("account_id").isNotNull() &
                F.col("jnj_id").isNotNull())
            .select("account_id", "jnj_id").distinct())

# CareSet HCPs: use the direct L0 OPAL mapping already created.
# careset_acc is already filtered to positive monthly records overlapping the window.
# CareSet has no product information and is kept separate from product-associated HCPs.
careset_hcp = (careset_acc.filter(
                   F.col("account_id").isNotNull() &
                   F.col("opal_child_id").isNotNull() &
                   (F.col("num_of_records") > 0))
              .select("account_id", F.col("opal_child_id").alias("jnj_id"))
              .distinct())

hcps = (hn_hcp.select("account_id", "jnj_id").distinct()
        .withColumn("role", F.lit("LE.AI H&N HCP"))
        .unionByName(writers.withColumn("role", F.lit("Product-associated HCP (IV/FASPRO/Lazcluze/Tagrisso)")))
        .unionByName(careset_hcp.withColumn("role", F.lit("CareSet Lung HCP (no product information)")))
        .join(seg_of, "account_id")
        .join(dmn_info.select("jnj_id", "specialty"), "jnj_id", "left")
        .cache())

sp = (hcps.groupBy("segment", "role", F.coalesce("specialty", F.lit("UNKNOWN")).alias("specialty"))
          .agg(F.countDistinct("jnj_id").alias("hcps")).toPandas())

print("1) Specialty mix")
print("Tagrisso claims included in product-associated HCP evidence:", INCLUDE_TAGRISSO_CLAIMS_IN_LUNG)
for (sg, rl), g in sp.groupby(["segment", "role"]):
    g = g.sort_values(["hcps", "specialty"], ascending=[False, True]).copy()
    total_hcps = g["hcps"].sum()
    g["pct"] = (100 * g["hcps"] / total_hcps).round(1) if total_hcps > 0 else 0.0
    print(f"\n{sg} | {rl} | {int(total_hcps):,} HCPs")
    print(g.head(8)[["specialty", "hcps", "pct"]].to_string(index=False))

# HCP overlap is based on the same account and L0 J&J ID.
# Product-associated overlap follows eligible activity and the claims flag.
dual = (hn_hcp.select("account_id", "jnj_id").distinct()
              .join(writers, ["account_id", "jnj_id"])
              .groupBy("account_id")
              .agg(F.countDistinct("jnj_id").alias("dual_hcps")))

nw = (writers.groupBy("account_id")
             .agg(F.countDistinct("jnj_id").alias("product_associated_hcps")))

nc = (careset_hcp.groupBy("account_id")
                 .agg(F.countDistinct("jnj_id").alias("careset_lung_hcps")))

dual_careset = (hn_hcp.select("account_id", "jnj_id").distinct()
                     .join(careset_hcp, ["account_id", "jnj_id"])
                     .groupBy("account_id")
                     .agg(F.countDistinct("jnj_id").alias("hn_careset_hcps")))

per = (ins.select("account_id", "segment", "hn_hcps")
          .join(nw, "account_id", "left")
          .join(dual, "account_id", "left")
          .join(nc, "account_id", "left")
          .join(dual_careset, "account_id", "left")
          .fillna(0, subset=["hn_hcps", "product_associated_hcps", "dual_hcps",
                            "careset_lung_hcps", "hn_careset_hcps"]))

print("\n2) HCPs per account")
(per.groupBy("segment")
    .agg(F.count("*").alias("accounts"),
         F.round(F.avg("hn_hcps"), 1).alias("avg_leai_hn_hcps"),
         F.round(F.avg("product_associated_hcps"), 1).alias("avg_product_associated_hcps"),
         F.round(F.avg("careset_lung_hcps"), 1).alias("avg_careset_lung_hcps"),
         F.sum("dual_hcps").alias("hn_product_overlap_hcps"),
         F.sum((F.col("dual_hcps") > 0).cast("int")).alias("accounts_with_hn_product_overlap"),
         F.sum("hn_careset_hcps").alias("hn_careset_overlap_hcps"),
         F.sum((F.col("hn_careset_hcps") > 0).cast("int")).alias("accounts_with_hn_careset_overlap"))
    .orderBy("segment").show(truncate=False))

print("3) Account type (dmn_cust)")
(ins.select("account_id", "segment")
    .join(dmn_info.select(F.col("jnj_id").alias("account_id"), "cust_type"), "account_id", "left")
    .groupBy(F.coalesce("cust_type", F.lit("UNKNOWN / NOT IN DMN_CUST")).alias("account_type"))
    .pivot("segment", seg_order).count().fillna(0)
    .orderBy(F.desc("`H&N + Lung`"), "account_type").show(20, False))

# Practice setting is available from the LE.AI H&N cohort only.
ps = (h.select(clean(pick(h, "NPI")).alias("npi"),
               clean(pick(h, "Practice Settings")).alias("practice_setting"))
       .filter("npi IS NOT NULL AND practice_setting IS NOT NULL")
       .groupBy("npi")
       .agg(F.min("practice_setting").alias("practice_setting")))

acc_ps = (hn_hcp.select("account_id", "npi").distinct()
                .join(ps, "npi")
                .groupBy("account_id", "practice_setting").count()
                .withColumn("rk", F.row_number().over(
                    Window.partitionBy("account_id").orderBy(F.desc("count"), "practice_setting")))
                .filter("rk = 1").select("account_id", "practice_setting"))

print("4) Practice setting (LE.AI, H&N accounts)")
(ins.filter("segment != 'Lung only'").select("account_id", "segment")
    .join(acc_ps, "account_id", "left")
    .groupBy(F.coalesce("practice_setting", F.lit("UNKNOWN")).alias("practice_setting"))
    .pivot("segment", ["H&N only", "H&N + Lung"]).count().fillna(0)
    .orderBy(F.desc("`H&N only`"), "practice_setting").show(20, False))

# MX care-setting profile.
# Retain all claims products, including Tagrisso, for descriptive reporting.
# This profile does not change Lung eligibility or the four account patient counts.
# Lazcluze and Tagrisso MX records are claim events, not assumed infusions.
# Modifier flags are retained as proxies, not confirmed 340B status.
# Use the same patient-ID validity rule as block 5.
mods = F.upper(F.coalesce(F.col("modifiers"), F.lit("")))
mxs = (df["mx"].withColumn("product", ryb_product_ndc(F.col("ndc")))
        .filter(F.col("product").isin(CLAIMS_LUNG_PRODUCTS))
        .select(claims_patient_id.alias("patient_id"),
                F.to_date("service_date").alias("day"), "ndc", "product",
                clean(F.col("rendering_npi")).alias("npi"),
                F.upper("data_source").alias("data_source"),
                (mods.contains("|TB|") | mods.contains("|UD|")).cast("int").alias("is_340b"))
        .filter(win & F.col("patient_id").isNotNull())
        .join(npi_jnj, "npi", "left"))
mxs = to_account(mxs, "jnj_id").join(seg_of, "account_id")

print("5) MX claim events by care setting and product (IV, FASPRO, Lazcluze, Tagrisso)")
print("Descriptive profile includes Tagrisso regardless of the claims evidence flag.")
print("Patients are distinct within each segment/product, not additive across products.")
print("Line percentages use retained raw claim lines with valid patient IDs.")
(mxs.groupBy("segment", "product")
    .agg(F.countDistinct(F.struct("patient_id", "day", "ndc")).alias("claim_events"),
         F.countDistinct("patient_id").alias("patients"),
         F.round(100 * F.avg((F.col("data_source") == "INSTITUTIONAL").cast("int")), 1).alias("pct_facility_lines"),
         F.round(100 * F.avg("is_340b"), 1).alias("pct_tb_ud_modifier_lines"))
    .orderBy("segment", "product").show(truncate=False))

# DDD purchase activity: IV + FASPRO + Lazcluze + Tagrisso.
# Anchor the latest week to qualifying purchases inside the analysis window.
# Filter by actual day first so the first partial week is not dropped.
ddd_window = (ddd.filter(
                  F.col("account_id").isNotNull() &
                  F.col("product").isin(LUNG_PRODUCTS) &
                  win &
                  (F.col("amount") > 0))
              .cache())

last_wk = ddd_window.agg(F.max("week")).collect()[0][0]

print("6) DDD purchase activity (IV + FASPRO + Lazcluze + Tagrisso), latest mapped purchase week:", last_wk)

if last_wk is not None:
    d13 = F.date_sub(F.lit(last_wk), 13 * 7)
    d26 = F.date_sub(F.lit(last_wk), 26 * 7)

    ad = (ddd_window.groupBy("account_id")
          .agg(F.min("week").alias("first_week"),
               F.countDistinct("week").alias("weeks_active"),
               F.sum(F.when(F.col("week") > d13, F.col("amount")).otherwise(0)).alias("last13"),
               F.sum(F.when((F.col("week") > d26) & (F.col("week") <= d13),
                            F.col("amount")).otherwise(0)).alias("prev13"))
          .join(seg_of, "account_id"))

    # "First observed" refers to the selected window, not lifetime adoption.
    (ad.groupBy("segment")
       .agg(F.count("*").alias("buying_accounts"),
            F.min("first_week").alias("earliest_observed_week"),
            F.round(F.avg("weeks_active"), 1).alias("avg_weeks_active"),
            F.sum((F.col("first_week") > d13).cast("int")).alias("first_observed_last_13w"),
            F.sum((F.col("last13") > F.col("prev13")).cast("int")).alias("higher_units_vs_prev13"),
            F.sum((F.col("last13") < F.col("prev13")).cast("int")).alias("lower_units_vs_prev13"),
            F.sum((F.col("last13") == 0).cast("int")).alias("no_positive_purchase_last_13w"))
       .orderBy("segment").show(truncate=False))
else:
    print("No positive mapped DDD purchases found inside the analysis window.")

## 10. Save to the Flow

#### What this cell does: create the output datasets (if missing) and write the results

- Set SAVE = True to write the results; False skips dataset creation and writing.
- Before writing, checks that the account output contains MX, RX, MX\_TAGRISSO, and RX\_TAGRISSO. Stops with an error if any are missing.
- Missing datasets are created automatically in the Flow, on the same connection as RYB\_DDD\_Sales\_copy.
- Existing datasets with the selected output names are written using the current results and schema. This cell does not append historical run snapshots.
- **Outputs:**
    - **HN\_LUNG\_ACCOUNT\_SEGMENTS\_L2L3\_52W****:** one row per mapped account in the segmentation universe, including its H&N/Lung segment, available account name and level, LE.AI H&N HCP count and two\-year patient volume, source\-specific Lung measures, DDD IV/FASPRO/Lazcluze/Tagrisso units, product mix, eligible Lung\-source count, and Chargemaster\-only/CareSet\-only Lung evidence flags.
    - **Claims patient counts:** the account output includes MX / RX for distinct non\-Tagrisso patients and MX\_TAGRISSO / RX\_TAGRISSO for separate distinct Tagrisso patients, per account over 52 weeks. RX counts retain the MX patient/product exclusion. These columns are not additive.
    - **HN\_LUNG\_RYBREVANT\_FASPRO\_DDD\_WEEKLY\_SEGMENTS\_L2L3\_52W****:** one row per account × week with signed FASPRO DDD units and the weekly segment. Unmapped DDD activity is retained under “Unmapped account.” Lung presence uses eligible non\-DDD evidence across the full analysis window, including CareSet, or positive IV/FASPRO/Lazcluze/Tagrisso DDD purchases in the preceding 13 weeks.
- The weekly output name is generated from SPLIT\_PRODUCTS; the name above reflects the current FASPRO\-only setting. Full\-window claims patient counts are not repeated in the weekly output.
- Tagrisso claims evidence follows INCLUDE\_TAGRISSO\_CLAIMS\_IN\_LUNG; separate Tagrisso patient counts remain available for accounts in the account output regardless of the flag. Tagrisso DDD/Xponent evidence is unaffected.
- CareSet contributes monthly Lung evidence without product attribution. Its record counts remain separate from sales units and patient counts.
- The intermediate claims\_patient\_counts\_52w table is not saved separately by this cell.
- Previously named output datasets are not renamed or deleted by this cell.



In [0]:
SAVE = True

# Account output includes all eligible Lung sources and separate claims counts:
# MX / RX: distinct non-Tagrisso patients per account over 52 weeks.
# MX_TAGRISSO / RX_TAGRISSO: separate distinct Tagrisso patients.
# RX counts retain the existing MX patient/product exclusion.
# Tagrisso patient counts remain available regardless of the claims evidence flag.
#
# Weekly output contains only the DDD products selected in SPLIT_PRODUCTS.
# Tagrisso claims evidence follows INCLUDE_TAGRISSO_CLAIMS_IN_LUNG.
# Tagrisso DDD/Xponent evidence remains unchanged.
split_label = "_".join(
    re.sub(r"[^A-Z0-9]+", "_", product.upper()).strip("_")
    for product in SPLIT_PRODUCTS
)

outputs = {
    "HN_LUNG_ACCOUNT_SEGMENTS_L2L3_52W": ins,
    f"HN_LUNG_{split_label}_DDD_WEEKLY_SEGMENTS_L2L3_52W": weekly,
}

if SAVE:
    # Check required patient-count columns before creating or writing outputs.
    required_cols = ["MX", "RX", "MX_TAGRISSO", "RX_TAGRISSO"]
    missing_cols = [c for c in required_cols if c not in ins.columns]
    if missing_cols:
        raise ValueError(
            f"Account output is missing required patient-count columns: {missing_cols}. "
            "Run the updated blocks 5, 6, and 8 before saving."
        )

    client = dataiku.api_client()
    project = client.get_default_project()
    conn = project.get_dataset(DATASETS["ddd"]).get_settings().get_raw()["params"]["connection"]
    existing = [d["name"] for d in project.list_datasets()]

    for name, d in outputs.items():
        if name not in existing:
            b = project.new_managed_dataset(name)
            b.with_store_into(conn)
            b.create()
            print("created", name)

        dkuspark.write_with_schema(dataiku.Dataset(name), d)
        print("saved", name)

    print("Account output includes patient-count columns:", required_cols)
    print("Tagrisso claims included in Lung evidence:", INCLUDE_TAGRISSO_CLAIMS_IN_LUNG)
else:
    print("SAVE = False: no datasets written.")

# Explanation of the Analysis

# Explanation of the Analysis
## Roll-up and segmentation rules: Rybrevant IV, FASPRO, Lazcluze, Tagrisso, and CareSet Lung evidence

### Scope

- **Products:** Rybrevant IV, Rybrevant FASPRO, Lazcluze, and Tagrisso.
- **Analysis window:** 52 weeks, **October 24, 2025 through October 22, 2026**, inclusive.
- **Account level:** OPAL-selected L2/L3 account. HCP/L0 and outlet/L1 mappings use the supplied `L2_L3`; L2 and L3 IDs can map to themselves under the shared lookup rules.
- **Additional Lung evidence:** CareSet Lung records and Chargemaster NSCLC encounters establish Lung presence without identifying a specific product.
- **Segmentation method:** Presence-based account classification. The code does not allocate sales using patient-distribution percentages or indication-share thresholds.
- **Claims reporting:** MX and RX report distinct non-Tagrisso patients. Separate `MX_TAGRISSO` and `RX_TAGRISSO` columns report distinct Tagrisso patients.
- **Tagrisso claims flexibility:** `INCLUDE_TAGRISSO_CLAIMS_IN_LUNG` controls whether Tagrisso MX/RX activity establishes Lung presence. Its current setting is `True`.
- **H&N timing:** LE.AI remains a two-year H&N cohort. Its treating-patient volume is not a 52-week measure.
- **Coverage:** The configured window does not guarantee that every source contains complete data through October 22, 2026.

## Shared lookups

### NPI to J&J ID: `dmn_cust_copy`

- Map `npi_id` to `jnj_id`.
- Exclude missing NPIs and J&J IDs.
- If an NPI has multiple J&J IDs, retain the record with the latest `efftv_dt`.
- Resolve effective-date ties using the smallest J&J ID in string sort order.
- This lookup does not independently apply active-status or end-date filters. Any such restrictions must already exist upstream.

### J&J ID to L2/L3 account: OPAL

Each ID receives at most one account mapping, using the following priority:

| Priority | ID type | Mapping rule |
|---|---|---|
| 1 | HCP/L0, OPAL column `0` | Use the supplied `L2_L3`; if multiple accounts remain, select the smallest account ID in string sort order. |
| 2 | Outlet/L1, OPAL column `1` | Use `OPAL_Affiliation_L1`; select the most frequent `L2_L3` mapping, breaking ties by account ID. |
| 3 | L2, OPAL column `2` | Map the ID to itself. |
| 4 | L3, OPAL column `3` | Map the ID to itself. |
| No match | Unresolved ID | Retain as unmapped in activity diagnostics; exclude from mapped account segmentation. Qualifying unmapped DDD sales remain in the weekly output. |

**Account-level label:** L2 if the selected account appears in OPAL column `2`; otherwise L3 if it appears in column `3`. Unresolved level labels are reported as `UNKNOWN` in the account output.

One mapping per ID prevents multiplication of source rows during the account join. It does not deduplicate patients across different accounts or sources.

### CareSet L0 to L2/L3 mapping

- **Source dataset:** `CARESET_LUNG_copy`.
- Map `opal_child_id` directly to OPAL HCP/L0 column `0`.
- Use the corresponding `L2_L3` account.
- If an L0 ID maps to multiple accounts, retain the smallest account ID in string sort order.
- Do not use NPI as a fallback mapping key.
- Retain NPI and month in the intermediate data for traceability.
- Retain unmapped records for diagnostics, but do not assign them to a mapped account segment.

## Product configuration

### Overall Lung products

`LUNG_PRODUCTS` contains:

- Rybrevant IV
- Rybrevant FASPRO
- Lazcluze
- Tagrisso

DDD and Xponent recognize these products from product names. MX and RX recognize them through the configured NDC mapping.

### Claims product groups

| Setting | Purpose |
|---|---|
| `CLAIMS_LUNG_PRODUCTS` | Retains all four products during MX/RX processing. |
| `CLAIMS_NON_TAGRISSO_PRODUCTS` | Defines the products counted in `MX` and `RX`: Rybrevant IV, FASPRO, and Lazcluze. |
| `CLAIMS_TAGRISSO_PRODUCTS` | Defines the product counted in `MX_TAGRISSO` and `RX_TAGRISSO`. |
| `INCLUDE_TAGRISSO_CLAIMS_IN_LUNG` | Controls whether Tagrisso claims contribute to Lung evidence. |

Setting the flag to `False` removes Tagrisso claims from eligible segmentation activity, but does not remove their separate patient counts from `claims_patient_counts_52w`.

**The flag does not remove Tagrisso from DDD, Xponent, or prior-DDD Lung evidence.**

## Rules by dataset

### 1. DDD

- **Identifier:** Buyer `jnj_id`, mapped through the shared OPAL lookup.
- **Products:** Rybrevant IV, FASPRO, Lazcluze, and Tagrisso, recognized from `prod_nm`.
- **Measure:** `equiv_uns`.
- **Date:** `caln_dt`.
- **Account Lung evidence:** Positive qualifying activity dated within the analysis window.
- **Weekly historical evidence:** Positive purchases of any configured Lung product in the preceding `LOOKBACK_WEEKS`, currently 13 weeks.
- **History retention:** Keep pre-window DDD history to support the earliest weekly lookbacks.
- **Product totals:** Report `DDD_IV`, `DDD_FASPRO`, `DDD_LAZCLUZE`, and `DDD_TAGRISSO` separately.

In the account activity table, `DDD` includes positive qualifying activity across all four products. Negative and zero activity rows are removed from the common `activity` table.

The separate `ddd` dataframe retains signed amounts. Weekly sales outputs and the all-buyer weekly trend use this dataframe, so their totals can differ from positive-only account activity totals.

### 2. Xponent

- **Identifier:** HCP `jnj_id`, mapped through OPAL.
- **Products:** Rybrevant IV, FASPRO, Lazcluze, and Tagrisso, recognized from `prod_nm`.
- **Measure:** Positive `equ_trx`.
- **Date:** `caln_dt`.
- **Signal:** Lung.
- **Tagrisso claims flag:** Does not affect Xponent.

### 3. MX claims

- **Identifier:** `rendering_npi` to J&J ID through `dmn_cust_copy`, then OPAL.
- **Product identification:** Match the configured NDCs after removing nonnumeric characters.

| Product | Configured NDCs |
|---|---|
| Rybrevant IV | `57894050100`, `57894050101` |
| Rybrevant FASPRO | `57894051001`, `57894051401`, `57894051501`, `57894052201` |
| Lazcluze | `57894008060`, `57894008090`, `57894024030`, `57894024099` |
| Tagrisso | `00310124830`, `00310125130`, `00310134930`, `00310135030`, `00310135095` |

These are the configured recognition values, not a statement that the mapping is exhaustive. NDC normalization does not automatically convert every possible 10-digit representation into an 11-digit NDC.

#### Patient identification and event preparation

- **Patient identifier:** Concatenate cleaned `dv1` and `dv2` with a separator.
- **Validity:** Both components must be present. Otherwise, `patient_id` is null.
- **Event deduplication:** Retain one record per patient × service date × NDC × rendering NPI.
- **Provider preservation:** Including NPI in deduplication preserves different provider associations before account mapping.
- **Date:** `service_date`.

#### Final MX patient counts

- **`MX`:** Distinct patients with qualifying Rybrevant IV, FASPRO, or Lazcluze MX activity at the selected account during the 52-week window.
- **`MX_TAGRISSO`:** Distinct patients with qualifying Tagrisso MX activity at the selected account during the same window.

Each patient is counted once per account within each reporting group, regardless of the number of dates, claims, or eligible products.

#### Activity and Lung evidence

The common `activity` table retains an event measure of one per retained MX event. This is used for activity diagnostics and presence checks, not as the final patient-count measure.

MX activity with a null patient ID does not contribute to claims-based Lung evidence. Tagrisso MX evidence additionally follows `INCLUDE_TAGRISSO_CLAIMS_IN_LUNG`.

### 4. RX claims

- **Identifier:** `prescriber_npi` to J&J ID through `dmn_cust_copy`, then OPAL.
- **Products:** Same configured NDC mapping as MX.
- **Patient identifier:** Same cleaned `dv1` and `dv2` construction and validity rule as MX.
- **Date:** `fill_date`.

#### MX exclusion

Exclude RX records when the same patient appears in MX for the same product.

- MX activity for one product does not exclude RX activity for another product.
- The exclusion uses available MX history, not only in-window MX records.
- The exclusion is not restricted to the same account.
- This is a patient/product exclusion rule, not exact matching of individual claim events or dates.

#### Deduplication and final RX counts

- **Deduplication:** One retained record per `claim_id` × product.
- **`RX`:** Distinct patients with retained non-Tagrisso RX activity at the selected account during the 52-week window.
- **`RX_TAGRISSO`:** Distinct patients with retained Tagrisso RX activity at the selected account during the same window.
- Both counts are calculated after the MX patient/product exclusion.

The common `activity` table retains one per retained RX event for diagnostics and presence checks. The final account columns use distinct patient counts instead.

RX activity with a null patient ID does not contribute to claims-based Lung evidence. Tagrisso RX evidence additionally follows the claims flag.

### 5. McKesson EMR

- **Identifier:** `physician_npi` to J&J ID through `dmn_cust_copy`, then OPAL.
- **Filter:** In-window alerts whose `alert_name` contains `RYB` or explicitly contains `LAZCLUZE`.
- **Measure:** One per qualifying alert row.
- **Date:** `diagnosis_date`.
- **Product classification:** Rybrevant alerts are classified as FASPRO, IV, or unspecified. Lazcluze-only alerts are classified as Lazcluze.
- **Combination alerts:** Count once, retaining the Rybrevant classification when both names appear.
- **Lung signal:** Any qualifying alert under the existing implementation.
- **H&N signal:** A qualifying Rybrevant alert whose diagnosis contains `HNSCC`, `HEAD`, or `NECK`.
- **Tagrisso:** No standalone Tagrisso alert rule has been added.

Because qualifying Rybrevant alerts also establish Lung presence, an H&N-diagnosis Rybrevant alert can contribute to both flags under the current logic. These are operational evidence rules, not confirmed indication attribution.

### 6. Chargemaster

- **Identifier:** NPI to J&J ID through `dmn_cust_copy`, then OPAL.
- **Filter:** Monthly rows whose reported period overlaps the analysis window.
- **Aggregation:** Sum encounters per NPI before account aggregation.
- **Lung measure:** Positive `# EncountersNSCLC`.
- **H&N measure:** Positive `# EncountersHNSCC`.
- **Product attribution:** None.
- **Boundary months:** Include overlapping monthly counts in full; do not prorate to individual days.

### 7. CareSet Lung claims

- **Dataset:** `CARESET_LUNG_copy`.
- **Identifier:** `opal_child_id`, mapped directly from L0 to OPAL `L2_L3`.
- **Measure:** Positive `num_of_records`.
- **Time field:** `month_start_date`.
- **Filter:** Include a month when its month-end is on or after the window start and its month-start is on or before the window end.
- **Boundary months:** Include the full monthly record count without daily proration.
- **Aggregation:** Sum records by selected account and original L0 ID, then aggregate at account level.
- **Signal:** Lung, including accounts with no other Lung source.
- **Product attribution:** None.
- **Activity dates:** `day` and `week` remain null because monthly overlap has already been applied.

CareSet record counts are not interpreted as unique patients, prescriptions, or sales units. The current code does not use CareSet to calculate an H&N-versus-Lung patient distribution.

### 8. LE.AI H&N

- **Identifier:** Supplied J&J ID; use the NPI-to-J&J lookup when the supplied ID is missing or a placeholder.
- **Account mapping:** OPAL, with `HN_ACCOUNT_SOURCE = "opal"`.
- **Measures:** Distinct H&N NPIs and two-year treating-patient volume.
- **Deduplication:** Retain the maximum patient volume per account × NPI, then sum across HCPs.
- **Signal:** H&N.
- **Timing:** Retain the two-year cohort; do not relabel its volume as 52-week patients.

`hn_pts_2y` is a sum of supplied HCP patient volumes. It is not a deduplicated account-level patient count.

## Distinct claims patient-count table

`claims_patient_counts_52w` contains one row per mapped account with valid in-window claims patients.

| Column | Definition |
|---|---|
| `MX` | Distinct non-Tagrisso MX patients per account over 52 weeks. |
| `RX` | Distinct non-Tagrisso RX patients per account over 52 weeks, after MX patient/product exclusion. |
| `MX_TAGRISSO` | Distinct Tagrisso MX patients per account over 52 weeks. |
| `RX_TAGRISSO` | Distinct Tagrisso RX patients per account over 52 weeks, after MX patient/product exclusion. |

Counts are calculated directly across the full window after account mapping. They are not sums of weekly or product-level distinct counts.

### Non-additivity

- A patient using both Tagrisso and another eligible product can appear in both reporting groups.
- A patient can appear in MX for one product and RX for another.
- A patient treated at multiple accounts can appear in multiple account rows.
- Therefore, adding the four columns does not produce a combined unique patient count.
- Summing an account-level column across accounts produces account-patient counts, not globally unique patients.

## Account segmentation

### Lung present

A mapped account has positive qualifying evidence from at least one of:

- DDD or Xponent for any configured Lung product.
- MX or RX for non-Tagrisso products.
- Tagrisso MX or RX when `INCLUDE_TAGRISSO_CLAIMS_IN_LUNG = True`.
- Qualifying McKesson alerts.
- Chargemaster NSCLC encounters.
- CareSet Lung records.

Dated sources must fall within the analysis window. Monthly sources use the overlap rules described above.

### H&N present

A mapped account has at least one of:

- LE.AI H&N cohort presence.
- Positive Chargemaster HNSCC encounters, when `CM_HNSCC_AS_HN = True`.
- A qualifying McKesson Rybrevant alert with an H&N diagnosis, when `MCKESSON_HN_DX_AS_HN = True`.

### Segment assignment

| H&N evidence | Lung evidence | Segment |
|---|---|---|
| Yes | Yes | H&N + Lung |
| Yes | No | H&N only |
| No | Yes | Lung only |

Accounts with neither signal are not added to the account universe.

“H&N only” means no qualifying Lung evidence was observed under the configured rules. It does not establish that an account has never treated Lung patients or used the products.

### Separation of reporting counts and eligibility

The four patient-count columns are joined after the H&N/Lung universe is established. Their availability does not independently override the segmentation flag.

If Tagrisso claims evidence is disabled:

- An H&N account can retain positive Tagrisso patient counts and remain H&N only.
- An account with only Tagrisso claims and no H&N or other eligible Lung evidence is excluded from the final account segmentation table.
- Its counts remain available in `claims_patient_counts_52w`, which is not separately saved by the current output block.

## Weekly DDD split

- **Products split:** Controlled by `SPLIT_PRODUCTS`, currently FASPRO only.
- **Row grain:** Account × week, aggregated across selected split products.
- **Week definition:** Monday of the transaction week.
- **Window rule:** Include weeks whose Monday falls inside the analysis window. This differs from account segmentation, which filters actual transaction dates.
- **Sales measure:** Sum signed DDD amounts for the selected products.
- **Lung present for a sale week:** Either:
  1. Positive eligible non-DDD Lung evidence anywhere in the full analysis window; or
  2. A positive DDD purchase for any product in `LUNG_PRODUCTS` during the preceding 13 weeks.
- **Lookback:** Excludes the current week and counts distinct prior purchase weeks.
- **H&N present:** Uses the same H&N universe as account segmentation.
- **Segments:** H&N + Lung, H&N only, Lung only, Unclassified, or Unmapped account.

Tagrisso DDD can establish prior-purchase Lung evidence even when Tagrisso claims evidence is disabled.

The weekly classification is retrospective. Non-DDD evidence later in the window can classify an earlier sale week.

A current-week DDD sale alone does not establish Lung presence under this weekly rule. Therefore, weekly and account-level segments can differ.

CareSet-only accounts without qualifying DDD activity do not create weekly sales rows.

The four full-window patient-count columns are not repeated in the weekly output.

## Profiles and interpretation

- **Source count:** Number of distinct eligible Lung sources in `activity`, including CareSet. Tagrisso is a product within a source, not an additional source.
- **Evidence breadth:** `n_lung_sources` measures source breadth, not validated confidence.
- **CareSet-only Lung evidence:** CareSet is the account’s only eligible Lung source; the account can still have H&N evidence.
- **Chargemaster-only Lung evidence:** Chargemaster is the account’s only eligible Lung source.
- **DDD product mix:** Positive account totals for IV, FASPRO, Lazcluze, and Tagrisso. Product combinations do not establish concurrent treatment or chronological switching.
- **HCP profiles:** Product-associated HCPs use eligible MX, RX, Xponent, and McKesson activity. Tagrisso claims follow the evidence flag. DDD buyers are not treated as HCPs.
- **CareSet HCPs:** Reported separately and not labeled as product writers.
- **MX care-setting profile:** Retains all configured claims products, including Tagrisso, for accounts in the segmentation table, regardless of the claims-evidence flag.
- **MX profile patients:** Distinct at segment × product level. These counts need not equal sums of account-level patient counts.
- **MX modifier profile:** TB/UD modifier flags are proxies, not confirmed 340B status.
- **DDD purchase-activity profile:** Uses positive purchases. “First observed” refers to the selected window, not lifetime adoption.
- **Source measures:** Keep DDD units, Xponent prescriptions, claims patient counts, McKesson alerts, Chargemaster encounters, and CareSet records separate.

## Output datasets

### `HN_LUNG_ACCOUNT_SEGMENTS_L2L3_52W`

One row per mapped account in the segmentation universe, including:

- Account ID, available name, and account level.
- Final segment and H&N/Lung presence flags.
- H&N HCP count, two-year patient-volume rollup, and H&N sources.
- DDD total and separate IV, FASPRO, Lazcluze, and Tagrisso totals.
- DDD product mix.
- Xponent, McKesson, Chargemaster, and CareSet measures.
- `MX`, `RX`, `MX_TAGRISSO`, and `RX_TAGRISSO` distinct patient counts.
- Eligible Lung-source count and source-only evidence flags.

### `HN_LUNG_RYBREVANT_FASPRO_DDD_WEEKLY_SEGMENTS_L2L3_52W`

One row per account × week with:

- FASPRO DDD units.
- Weekly segment.
- Account level.
- H&N evidence.
- Full-window non-DDD Lung-source evidence.
- Prior-DDD purchase-week count.

Qualifying unmapped sales remain included. The dataset name changes automatically if `SPLIT_PRODUCTS` changes.

### Save behavior

- `SAVE = True` writes the two configured datasets.
- Missing datasets are created on the connection used by the DDD input.
- The account output is checked for all four required claims patient-count columns before writing.
- The write uses the current dataframe schema.
- The block does not append historical run snapshots or separately persist `claims_patient_counts_52w`.

## Results and coverage limitations

### Results must come from the updated run

The earlier counts of **1,857 H&N only**, **1,238 H&N + Lung**, and **686 Lung only** belong to a previous run. They must not be presented as results of the revised Tagrisso, Lazcluze, CareSet, and distinct-patient-count implementation.

Use the newly executed `acct_seg`, `ins`, and summary outputs for final counts.

### Source coverage

The previously supplied activity summary showed latest activity-week labels of:

- **DDD and Xponent:** September 21, 2026.
- **MX and RX:** August 31, 2026.
- **McKesson:** September 21, 2026.

These are prior observed labels, not verified coverage results from the revised run. Rerun the coverage checks and activity summaries.

Minimum and maximum dates do not prove continuous coverage, product-specific coverage, or completeness of the latest month.

### Additional interpretation limitations

- Monthly boundary periods are included in full.
- RX exclusion uses available MX history, including records outside the analysis window.
- RX deduplication retains one record per claim ID and product. Conflicting duplicate rows are not resolved by a separate business-priority rule.
- Both patient-ID components must be valid for claims patient counting and claims-based Lung evidence.
- Patients can overlap across accounts, product groups, and sources.
- Account-name coverage depends on names available in the LE.AI cohort.
- Different L2/L3 assignments within a health system remain separate selected accounts.
- Positive-only account DDD activity and signed weekly DDD totals use different aggregation rules.
- The segmentation identifies historical evidence overlap. It does not prove the indication of an individual purchase or allocate FASPRO sales between H&N and Lung.